# Notebook 0 — BDHS 2022 Child Recode: Topic Discovery and Data Audit

**Purpose:** build evidence for the research team to choose a prediction outcome before explanation experiments. This notebook uses no explainers and does not choose or freeze an outcome.

**Privacy:** run in the team's own Google Colab account. The Stata file is read from the team's Drive into that runtime. No record-level rows, individual identifiers, rare variable combinations, or sensitive responses are displayed or exported. Only aggregate metadata and summaries are saved under audit_outputs/. The notebook never calls an AI service.

**Candidate set:** these are outcomes identified from our current research discussion, not all viable BDHS Child Recode outcomes.

**Use:** review every TODO / REVIEW item. Variable names alone are not evidence; target construction and survey roles fail closed when labels or value labels do not match.

**Notebook version:** 0.2.1 (corrective review; separate from research framework v1.1).


**Important boundary:** this is the *outcome-selection and data-audit notebook*, not the final SHAP/LIME/Integrated Gradients experiment notebook. It deliberately does not train the thesis models, calculate explainer agreement, or select an outcome automatically. Those steps begin only after the two-person decision gate in the final section is completed. The preliminary model screen is also blocked by default until outcome definitions, leakage review, prediction-time boundary, estimand, and the unweighted-screen decision are explicitly confirmed.

**Verified file snapshot (the DTA uploaded with this project, checked 5 Nov 2026):** 8,784 rows × 1,238 variables; `v000=BD8`; `v007=2022`. The uploaded file's labels identify `v001` as cluster number, `v021` as primary sampling unit, `v022` as sample strata for sampling errors, `v023` as stratification used in sample design, and `v005` as women's individual sample weight. `v001` and `v021` both had 674 distinct values in the supplied file. These are file-level observations, **not** a substitute for confirming survey design definitions from the official BDHS documentation. The notebook recomputes these checks at runtime and never trusts this note over the live file.


## Corrective review — v0.2.2

This revision adds source cross-checks and closes issues found in a third review:

- Distinguishes fully completed, partial, failed, and blocked predictive-feasibility runs; records skipped folds and their reasons.
- Cross-checks the DTA's embedded variable labels and value-label counts against the separately supplied variable-dictionary CSV when available.
- Imports the literature matrix as an evidence inventory, reports its verification/read-level limitations, and does **not** treat its entries as independently verified papers.
- Makes the provisional anthropometry denominator and morbidity-question universe limitations more explicit.
- Records a manual code-revision identifier in the run manifest (not a cryptographic hash of the notebook source) and prevents a partially evaluated model from being called completed.

This remains a topic-discovery/data-audit notebook, not the final thesis experiment. It does not select an outcome, establish survey representativeness, or run explainers.

## 1 — Environment and reproducibility

Pin the Stata reader, mount the team's private Google Drive, and write aggregate audit outputs persistently under MyDrive/BDHS/audit_outputs. Record runtime/package versions, set the random seed, and record open team decisions. A run ID identifies one execution; the reproducibility identity includes dataset hash, notebook version, Git commit when available, and configuration.


In [ ]:
%pip -q install pyreadstat==1.3.7 psutil


In [ ]:
import os, sys, json, re, uuid, hashlib, random, platform, subprocess
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError

import numpy as np
import pandas as pd
import pyreadstat
import sklearn
import psutil
import matplotlib.pyplot as plt
from IPython.display import display
from google.colab import drive
drive.mount("/content/drive")

RANDOM_SEED = 20261030
NOTEBOOK_VERSION = "0.2.2"
NOTEBOOK_CODE_REVISION = "BDHS_TOPIC_AUDIT_0.2.2_source_crosschecks_partial_fold_status_r1"
RUN_TIMESTAMP_UTC = datetime.now(timezone.utc).isoformat()
RUN_ID = f"BDHS0-RUN-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}-{uuid.uuid4().hex[:8]}"
AUDIT_ID = RUN_ID
AUDIT_TIMESTAMP_UTC = RUN_TIMESTAMP_UTC
CV_FOLDS = 5

# TODO/REVIEW: set after checking the BDHS dictionary/report and recording both researchers' decision.
TEAM_CONFIRMED_PSU_VARIABLE = None  # allowed values: None, "v001", "v021"
PREDICTION_TIME_DEFINITION = "OPEN: define the time point and which predictors are available at or before it."
PREDICTION_POPULATION_DECISION = "OPEN: define the target population/estimand (e.g., sample-level classification, not population prevalence)."
SURVEY_WEIGHT_MODELING_DECISION = "OPEN: confirm that this preliminary feasibility screen is intentionally unweighted."
TEAM_CONFIRMED_PREDICTION_TIME = False
TEAM_CONFIRMED_PREDICTION_POPULATION = False
TEAM_CONFIRMED_UNWEIGHTED_PILOT = False
RUN_PREDICTIVE_FEASIBILITY_SCREEN = True  # execution still requires every confirmation gate below.

# Add candidate names only after both researchers have reviewed the relevant audit.
# Example syntax (do not copy blindly): {"stunting"}
TEAM_REVIEWED_OUTCOME_DEFINITIONS = set()
TEAM_CONFIRMED_OUTCOME_DEFINITIONS = set()  # both researchers
TEAM_REVIEWED_LEAKAGE_AUDITS = set()
TEAM_CONFIRMED_LEAKAGE_AUDITS = set()  # both researchers
if not TEAM_CONFIRMED_OUTCOME_DEFINITIONS.issubset(TEAM_REVIEWED_OUTCOME_DEFINITIONS):
    raise ValueError("Confirmed outcome definitions must also be marked reviewed.")
if not TEAM_CONFIRMED_LEAKAGE_AUDITS.issubset(TEAM_REVIEWED_LEAKAGE_AUDITS):
    raise ValueError("Confirmed leakage audits must also be marked reviewed.")
VALID_CANDIDATE_NAMES = {
    "stunting","wasting","underweight","fever","diarrhea","ari_symptoms",
    "child_anemia","full_age_appropriate_immunisation","mortality_related"
}
for decision_name, decision_set in [
    ("TEAM_REVIEWED_OUTCOME_DEFINITIONS", TEAM_REVIEWED_OUTCOME_DEFINITIONS),
    ("TEAM_CONFIRMED_OUTCOME_DEFINITIONS", TEAM_CONFIRMED_OUTCOME_DEFINITIONS),
    ("TEAM_REVIEWED_LEAKAGE_AUDITS", TEAM_REVIEWED_LEAKAGE_AUDITS),
    ("TEAM_CONFIRMED_LEAKAGE_AUDITS", TEAM_CONFIRMED_LEAKAGE_AUDITS),
]:
    unknown = set(decision_set) - VALID_CANDIDATE_NAMES
    if unknown:
        raise ValueError(f"{decision_name} contains unknown candidate names: {sorted(unknown)}")
if not isinstance(TEAM_CONFIRMED_PREDICTION_TIME, bool):
    raise TypeError("TEAM_CONFIRMED_PREDICTION_TIME must be True or False.")
if not isinstance(TEAM_CONFIRMED_PREDICTION_POPULATION, bool):
    raise TypeError("TEAM_CONFIRMED_PREDICTION_POPULATION must be True or False.")
if not isinstance(TEAM_CONFIRMED_UNWEIGHTED_PILOT, bool):
    raise TypeError("TEAM_CONFIRMED_UNWEIGHTED_PILOT must be True or False.")
if TEAM_CONFIRMED_PSU_VARIABLE not in {None, "v001", "v021"}:
    raise ValueError("TEAM_CONFIRMED_PSU_VARIABLE must be None, 'v001', or 'v021'.")

GOOGLE_DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/BDHS")
GOOGLE_DRIVE_PROJECT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = GOOGLE_DRIVE_PROJECT_DIR / "audit_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Place the two supplied reference CSVs here, or edit these paths.
VARIABLE_DICTIONARY_PATH = GOOGLE_DRIVE_PROJECT_DIR / "reference" / "variable_dictionary_metadata.csv"
LITERATURE_MATRIX_PATH = GOOGLE_DRIVE_PROJECT_DIR / "reference" / "literature_matrix_hybrid_master_v3.csv"
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

def git_commit_if_available():
    try:
        result=subprocess.run(["git","rev-parse","HEAD"],cwd=str(Path.cwd()),capture_output=True,text=True,timeout=5,check=True)
        return result.stdout.strip()
    except Exception:
        return None

def package_version(name):
    try: return version(name)
    except PackageNotFoundError: return "not-installed"

RUNTIME_INFO = {
    "python":sys.version.split()[0],"platform":platform.platform(),
    "colab_release_tag":os.environ.get("COLAB_RELEASE_TAG"),
    "colab_backend_version":os.environ.get("COLAB_BACKEND_VERSION"),
    "cpu_count":os.cpu_count(),
    "available_memory_bytes_at_start":int(psutil.virtual_memory().available),
    "packages":{"pandas":pd.__version__,"numpy":np.__version__,
        "scikit_learn":sklearn.__version__,"pyreadstat":pyreadstat.__version__,
        "matplotlib":package_version("matplotlib"),"psutil":package_version("psutil"),
        "scipy":package_version("scipy"),"ipython":package_version("ipython")},
}
REPRO_CONFIG = {
    "random_seed":RANDOM_SEED,"requested_grouped_cv_folds":CV_FOLDS,"notebook_code_revision":NOTEBOOK_CODE_REVISION,
    "primary_input_filename":"BDKR81FL.DTA","output_directory":str(OUTPUT_DIR),"output_persistence":"Google Drive",
    "candidate_outcomes_scope":"Current research-discussion candidates; not exhaustive.",
    "pilot_predictor_policy":"Dictionary-label screened general-purpose policy; not final thesis feature set.",
    "preliminary_models":["LogisticRegression","RandomForestClassifier"],
    "preliminary_model_parameters":{"logistic_regression":{"max_iter":500},"random_forest":{"n_estimators":120,"min_samples_leaf":3,"n_jobs":1}},
    "run_predictive_feasibility_screen_requested":RUN_PREDICTIVE_FEASIBILITY_SCREEN,"survey_weights_used_in_predictability_metrics":False,"unweighted_pilot_decision_confirmed":TEAM_CONFIRMED_UNWEIGHTED_PILOT,
    "prediction_time_definition":PREDICTION_TIME_DEFINITION,"prediction_time_confirmed":TEAM_CONFIRMED_PREDICTION_TIME,
    "prediction_population_decision":PREDICTION_POPULATION_DECISION,"prediction_population_confirmed":TEAM_CONFIRMED_PREDICTION_POPULATION,
    "survey_weight_modeling_decision":SURVEY_WEIGHT_MODELING_DECISION,
    "team_confirmed_psu_variable":TEAM_CONFIRMED_PSU_VARIABLE,
    "grouped_feasibility_policy":"If label-verified v001 and v021 induce identical partitions, run a provisional grouped screen on their joint equivalence partition without selecting a canonical PSU.",
    "anthropometric_bounds_100x":{"hw70_HAZ":[-600,600],"hw71_WAZ":[-600,500],"hw72_WHZ":[-500,500]},
    "explainers_used":[],"display_or_export_raw_records":False,
    "team_reviewed_outcome_definitions":sorted(TEAM_REVIEWED_OUTCOME_DEFINITIONS),
    "team_confirmed_outcome_definitions":sorted(TEAM_CONFIRMED_OUTCOME_DEFINITIONS),
    "team_reviewed_leakage_audits":sorted(TEAM_REVIEWED_LEAKAGE_AUDITS),
    "team_confirmed_leakage_audits":sorted(TEAM_CONFIRMED_LEAKAGE_AUDITS),
}
print(json.dumps({"run_id":RUN_ID,"timestamp_utc":RUN_TIMESTAMP_UTC,"runtime":RUNTIME_INFO,"configuration":REPRO_CONFIG},indent=2))


## 2 — Dataset verification

Mount the team's Drive and set the exact path to BDKR81FL.DTA. The notebook hashes the file, reads it locally, and checks v000, v007, and child-recode markers from the actual file metadata and labels. A filename match alone never validates the survey.

In [ ]:
# Drive was mounted in the environment/reproducibility section.

# TODO: point this to the team's existing Drive copy. Never put raw data in audit_outputs/.
DTA_PATH = Path("/content/drive/MyDrive/BDHS/BDKR81DT/BDKR81FL.DTA")
if DTA_PATH.name.upper() != "BDKR81FL.DTA":
    raise ValueError("Primary input must be BDKR81FL.DTA. Update DTA_PATH; do not silently switch files.")
if not DTA_PATH.is_file():
    raise FileNotFoundError(f"Input not found at {DTA_PATH}. Edit DTA_PATH to the team's Drive location.")

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

FILE_SIZE_BYTES = DTA_PATH.stat().st_size
DATASET_SHA256 = sha256_file(DTA_PATH)
REFERENCE_INPUT_FILES = {}
for ref_name, ref_path in {
    "variable_dictionary_metadata": VARIABLE_DICTIONARY_PATH,
    "literature_matrix": LITERATURE_MATRIX_PATH,
}.items():
    REFERENCE_INPUT_FILES[ref_name] = {
        "path": str(ref_path),
        "available": bool(ref_path.is_file()),
        "size_bytes": int(ref_path.stat().st_size) if ref_path.is_file() else None,
        "sha256": sha256_file(ref_path) if ref_path.is_file() else None,
    }
REPRO_CONFIG["reference_input_files"] = REFERENCE_INPUT_FILES
CODE_GIT_COMMIT=git_commit_if_available()
_CONFIG_IDENTITY={"dataset_sha256":DATASET_SHA256,"reference_input_files":REFERENCE_INPUT_FILES,
                  "notebook_version":NOTEBOOK_VERSION,
                  "code_git_commit":CODE_GIT_COMMIT,"configuration":REPRO_CONFIG}
CONFIGURATION_IDENTITY_SHA256=hashlib.sha256(
    json.dumps(_CONFIG_IDENTITY,sort_keys=True,separators=(",",":"),default=str).encode("utf-8")
).hexdigest()
EXPERIMENT_CONFIG_ID="BDHS0-CFG-"+CONFIGURATION_IDENTITY_SHA256[:16]
# No rows are previewed, printed, or exported.
data, stata_meta = pyreadstat.read_dta(str(DTA_PATH), apply_value_formats=False, user_missing=True)
USER_MISSING_CODES = getattr(stata_meta, "missing_user_values", {}) or {}
USER_MISSING_COUNTS, USER_MISSING_MASKS = {}, {}
for variable, codes in USER_MISSING_CODES.items():
    if variable in data.columns:
        mask = data[variable].isin(list(codes or []))
        USER_MISSING_MASKS[variable] = mask.astype(bool)
        USER_MISSING_COUNTS[variable] = int(mask.sum())
        if mask.any():
            data.loc[mask, variable] = np.nan

def variable_label(variable):
    labels = getattr(stata_meta,"column_names_to_labels",None) or getattr(stata_meta,"variable_to_label",{}) or {}
    return str(labels.get(variable,"") or "")

def value_labels(variable):
    return stata_meta.variable_value_labels.get(variable, {}) or {}

def label_matches(variable, pattern):
    label = variable_label(variable)
    return bool(label) and bool(re.search(pattern, label, flags=re.IGNORECASE))

def aggregate_code_counts(series, limit=20):
    counts = series.astype("string").fillna("<missing>").value_counts(dropna=False).head(limit)
    return [{"code": str(index), "count": int(count)} for index, count in counts.items()]

survey_observations = {}
for variable in ["v000", "v007"]:
    if variable in data.columns:
        survey_observations[variable] = {
            "variable_label": variable_label(variable),
            "aggregate_value_counts": aggregate_code_counts(data[variable]),
            "distinct_nonmissing_values": int(data[variable].nunique(dropna=True)),
        }
    else:
        survey_observations[variable] = {"status": "not present"}

year_label_ok = "v007" in data.columns and label_matches("v007", r"year.*interview|interview.*year")
year_values = []
if year_label_ok:
    numeric_years = pd.to_numeric(data["v007"], errors="coerce").dropna()
    year_values = sorted({int(x) for x in numeric_years.unique() if float(x).is_integer()})
survey_year = year_values[0] if len(year_values) == 1 else (year_values if year_values else None)
v000_values = sorted({str(x).strip().upper() for x in data["v000"].dropna().unique() if str(x).strip()}) if "v000" in data else []
v000_label_ok = "v000" in data.columns and label_matches("v000", r"country.*(?:code.*)?phase|country.*phase")
country_phase_value_ok = bool(v000_values) and all(re.fullmatch(r"BD[0-9]+", v) for v in v000_values)
child_recode_markers = {
    "bidx_birth_column_label": label_matches("bidx", r"birth.*column|birth.*index|index.*birth"),
    "b19_child_age_months_label": label_matches("b19", r"age.*months"),
    "h11_diarrhea_label": label_matches("h11", r"diarr?hoea|diarrhea"),
    "h22_fever_label": label_matches("h22", r"fever"),
    "hw70_height_for_age_label": label_matches("hw70", r"height.*age|height/age"),
    "hw71_weight_for_age_label": label_matches("hw71", r"weight.*age|weight/age"),
    "hw72_weight_for_height_label": label_matches("hw72", r"weight.*height|weight/height"),
}
health_markers = sum(child_recode_markers[k] for k in [
    "h11_diarrhea_label","h22_fever_label","hw70_height_for_age_label",
    "hw71_weight_for_age_label","hw72_weight_for_height_label"
])
expected_child_recode_marker = child_recode_markers["b19_child_age_months_label"] and health_markers >= 2
expected_bdhs_2022 = survey_year == 2022 and v000_label_ok and country_phase_value_ok and expected_child_recode_marker
verification_issues = []
if not year_label_ok: verification_issues.append("v007 label does not identify interview year.")
elif survey_year != 2022: verification_issues.append(f"v007 does not uniquely show 2022: {survey_year!r}.")
if not v000_label_ok: verification_issues.append("v000 label does not identify country and phase.")
if not country_phase_value_ok: verification_issues.append(f"v000 values do not verify Bangladesh phase codes: {v000_values!r}.")
verification_notes = []
if not child_recode_markers["bidx_birth_column_label"]:
    verification_notes.append(f"bidx actual label does not confirm birth column/index: {variable_label('bidx')!r}; it is not used as proof of recode type.")
if not child_recode_markers["b19_child_age_months_label"]:
    verification_issues.append("b19 label does not confirm child age in months.")
if health_markers < 2: verification_issues.append(f"Fewer than two KR health/anthropometry labels verify ({health_markers}).")
DATASET_METADATA = {
    "run_id": RUN_ID, "notebook_version": NOTEBOOK_VERSION,
    "input_path": str(DTA_PATH.resolve()), "filename": DTA_PATH.name,
    "file_size_bytes": int(FILE_SIZE_BYTES), "sha256": DATASET_SHA256,
    "rows": int(len(data)), "columns": int(data.shape[1]),
    "stata_file_label": str(getattr(stata_meta, "file_label", "") or ""),
    "stata_file_encoding": str(getattr(stata_meta, "file_encoding", "") or ""),
    "survey_metadata_observed": survey_observations,
    "verification_issues": verification_issues,
    "verification_notes": verification_notes,
    "v000_label_verified_as_country_phase": bool(v000_label_ok),
    "v000_values_verified_as_bangladesh_phase_codes": bool(country_phase_value_ok),
    "user_defined_missing_code_counts_by_variable": USER_MISSING_COUNTS,
    "survey_year_inferred_from_v007": survey_year, "v000_country_phase_values": v000_values,
    "v000_label": variable_label("v000") if "v000" in data else None,
    "v007_label": variable_label("v007") if "v007" in data else None,
    "child_recode_label_checks": child_recode_markers,
    "appears_to_be_expected_bdhs_2022_child_recode": bool(expected_bdhs_2022),
    "verification_rule": "Require label-verified v007=2022; label-verified v000 Bangladesh phase codes; label-verified b19 child age in months; and at least two KR health/anthropometry labels. bidx is separately reported and not assumed. Filename is not proof.",
    "raw_dataset_copied_to_output_directory": False,
}
(OUTPUT_DIR / "dataset_metadata.json").write_text(json.dumps(DATASET_METADATA, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
print(json.dumps(DATASET_METADATA, indent=2, ensure_ascii=False, default=str))
if verification_issues:
    raise RuntimeError("Dataset verification unresolved; review saved metadata and stop before outcome analysis.")


## 2A — Uploaded-file fingerprint cross-check

The following cell checks the live file against the uploaded snapshot described above. A mismatch is a **review warning**, not an automatic failure: it may mean the team selected a different survey file/version. Do not proceed until the difference is explained. No record-level rows or identifiers are displayed.


In [ ]:
# Aggregate-only cross-check against the exact DTA uploaded to this project on 5 Nov 2026.
EXPECTED_SNAPSHOT = {
    "rows": 8784,
    "columns": 1238,
    "file_size_bytes": 12160116,
    "sha256": "df91e8988c6a6522b71860bb5d9f3fdfb6210ee880b0c1d214569d78096a63fa",
    "v000_values": ["BD8"],
    "v007_values": [2022],
    "v001_distinct": 674,
    "v021_distinct": 674,
}
observed_snapshot = {
    "rows": int(data.shape[0]),
    "columns": int(data.shape[1]),
    "file_size_bytes": int(FILE_SIZE_BYTES),
    "sha256": DATASET_SHA256,
    "v000_values": sorted(data["v000"].dropna().astype(str).str.strip().unique().tolist()) if "v000" in data else [],
    "v007_values": sorted(pd.to_numeric(data["v007"], errors="coerce").dropna().unique().tolist()) if "v007" in data else [],
    "v001_distinct": int(data["v001"].nunique(dropna=True)) if "v001" in data else None,
    "v021_distinct": int(data["v021"].nunique(dropna=True)) if "v021" in data else None,
}
snapshot_checks = {
    key: observed_snapshot.get(key) == expected
    for key, expected in EXPECTED_SNAPSHOT.items()
}
print("Observed aggregate fingerprint:")
print(json.dumps(observed_snapshot, indent=2, default=str))
print("Snapshot checks:")
print(json.dumps(snapshot_checks, indent=2))
if not all(snapshot_checks.values()):
    print("REVIEW REQUIRED: the live DTA differs from the uploaded project snapshot. Confirm survey phase, recode type, file version and hash before interpreting results.")
else:
    print("Snapshot match. This verifies file identity characteristics only; it does not confirm the substantive outcome or survey-design decisions.")

# Verify whether v001 and v021 encode the same partition, not merely the same number of clusters.
def same_partition(left, right):
    pair = pd.DataFrame({"left": left, "right": right}).dropna()
    if pair.empty:
        return False
    # A one-to-one mapping in both directions means the two variables induce the same grouping.
    return (pair.groupby("left", dropna=False)["right"].nunique().max() == 1
            and pair.groupby("right", dropna=False)["left"].nunique().max() == 1)

if {"v001", "v021"}.issubset(data.columns):
    print("v001-v021 same observed partition:", bool(same_partition(data["v001"], data["v021"])))
else:
    print("v001-v021 same observed partition: not testable; one or both variables absent")


## 3 — Variable inventory and survey-role checks

Write one aggregate row per variable: label, storage type, unique-value count, missing count, and missing percentage. Survey roles are checked against the labels present in this file. A missing or mismatched label is reported as unverified; no alternate field is silently substituted.

In [ ]:
inventory_rows = []
for variable in data.columns:
    series = data[variable]
    inventory_rows.append({
        "variable": str(variable), "variable_label": variable_label(variable),
        "data_type": str(series.dtype), "unique_nonmissing_values": int(series.nunique(dropna=True)),
        "missing_count": int(series.isna().sum()),
        "system_missing_count": max(0,int(series.isna().sum())-USER_MISSING_COUNTS.get(variable,0)),
        "user_defined_missing_code_count": USER_MISSING_COUNTS.get(variable,0),
        "missing_percentage": float(series.isna().mean() * 100),
    })
variable_inventory = pd.DataFrame(inventory_rows)
variable_inventory.to_csv(OUTPUT_DIR / "variable_inventory.csv", index=False)

SURVEY_ROLE_SPECS = {
    "v000": ("country / survey phase", r"country.*code.*phase|country.*phase"),
    "v007": ("survey year", r"year.*interview|interview.*year"),
    "v001": ("cluster / PSU candidate", r"cluster"),
    "v021": ("PSU candidate", r"primary sampling unit|cluster"),
    "v022": ("strata candidate", r"stratification|strata"),
    "v023": ("strata candidate", r"stratification|strata"),
    "v005": ("sampling weight", r"sample weight"),
    "v002": ("household identifier component", r"household number"),
    "v003": ("mother/respondent identifier component", r"respondent.*line number|line number.*respondent"),
    "caseid": ("mother/respondent identifier", r"respondent.*id|case identification"),
    "b16": ("child identifier candidate", r"child.*line number|line number.*child"),
    "bidx": ("birth column / birth index candidate", r"birth.*column|birth.*index|index.*birth"),
    "b5": ("child alive status", r"alive"),
    "b19": ("child age in months", r"age.*months"),
    "hw13": ("anthropometry measurement status", r"result of measurement.*height.*weight"),
}
survey_role_rows = []
for variable, (role, expected_pattern) in SURVEY_ROLE_SPECS.items():
    present = variable in data.columns
    actual_label = variable_label(variable) if present else ""
    verified = present and bool(re.search(expected_pattern, actual_label, flags=re.IGNORECASE))
    survey_role_rows.append({
        "variable": variable, "role_under_review": role, "present": bool(present),
        "actual_variable_label": actual_label if present else "NOT PRESENT",
        "label_check": "verified" if verified else ("label mismatch" if present else "not present"),
        "expected_label_pattern": expected_pattern,
        "unique_nonmissing_values": int(data[variable].nunique(dropna=True)) if present else None,
    })
survey_role_inventory = pd.DataFrame(survey_role_rows)
survey_role_inventory.to_csv(OUTPUT_DIR / "survey_variable_role_checks.csv", index=False)
display(variable_inventory.head(12))
display(survey_role_inventory)
print(f"Saved {len(variable_inventory):,} aggregate variable summaries. No dataset rows were displayed.")


## 4 - Stage A: Candidate outcome discovery

The initial outcomes below are candidate outcomes identified from our current research discussion, not a complete list of viable BDHS Child Recode outcomes. Each recipe must be supported by the actual file labels and, where needed, the BDHS dictionary, questionnaire, and report. Missing or mismatched evidence leaves a candidate pending; no alternative variable is silently substituted.

Stage A constructs auditable outcome labels and reports eligibility, missing or invalid codes, sample counts, prevalence, leakage findings, and cluster support without using predictive model results.

In [ ]:
def verified_age_alive_mask(min_age, max_age):
    if "b19" not in data.columns or not label_matches("b19", r"age.*months"):
        return None, False, "Age eligibility cannot be computed: b19 label does not confirm age in months."
    age_mask = pd.to_numeric(data["b19"], errors="coerce").between(min_age, max_age, inclusive="both")
    if "b5" not in data.columns or not label_matches("b5", r"alive"):
        return None, False, "Living-child eligibility cannot be computed: b5 label does not confirm alive status."
    yes_codes = [code for code, label in value_labels("b5").items() if re.match(r"\s*(yes|alive)\b", str(label), re.I)]
    if len(yes_codes) != 1:
        return None, False, "Living-child eligibility needs one Yes/alive category in b5 value labels."
    return age_mask & data["b5"].isin(yes_codes), True, f"b19 label-verified age {min_age}-{max_age} months and b5 label-verified living child."

def yes_no_codes(variable):
    labels = value_labels(variable)
    yes = [code for code, label in labels.items() if re.match(r"\s*yes\b", str(label), re.I)]
    no = [code for code, label in labels.items() if re.match(r"\s*no\b", str(label), re.I)]
    return yes, no

def build_binary_question(variable, expected_pattern, eligible, eligibility_ok, eligibility_note):
    if variable not in data.columns:
        return None, False, f"{variable} is not present."
    if not label_matches(variable, expected_pattern):
        return None, False, f"{variable} label mismatch; actual label: {variable_label(variable)}"
    yes, no = yes_no_codes(variable)
    if len(yes) < 1 or len(no) != 1:
        return None, False, f"{variable} value labels must contain one or more affirmative Yes categories and exactly one No category."
    if not eligibility_ok or eligible is None:
        return None, False, eligibility_note
    target = pd.Series(np.nan, index=data.index, dtype="float64")
    target.loc[eligible & data[variable].isin(yes)] = 1
    target.loc[eligible & data[variable].isin(no)] = 0
    affirmative_labels = [str(value_labels(variable).get(code, code)) for code in yes]
    return target, True, f"{variable} label/value labels verified; affirmative categories={affirmative_labels}; eligibility: {eligibility_note}"

def verified_anthropometry_status_masks():
    if "hw13" not in data.columns or not label_matches("hw13",r"result of measurement.*height.*weight"):
        return None,None,False,"hw13 absent or its actual label does not verify measurement status."
    labels=value_labels("hw13")
    measured=[c for c,l in labels.items() if re.fullmatch(r"\s*measured\s*",str(l),re.I)]
    no_measure=[c for c,l in labels.items() if re.fullmatch(r"\s*no measurement found in household\s*",str(l),re.I)]
    if len(measured)!=1 or len(no_measure)!=1:
        return None,None,False,"hw13 must have unique measured and no-measurement-found labels; review the actual dictionary."
    measured_mask=data["hw13"].isin(measured)
    # Provisional file-defined sample proxy; not assumed equal to FR386 eligibility.
    cohort=data["hw13"].notna() & data["hw13"].isin(list(labels)) & ~data["hw13"].isin(no_measure)
    return cohort,measured_mask,True,"Known hw13 categories excluding 'no measurement found in household'; this operational cohort needs documentation review."

def build_anthropometric_target(variable,label_pattern,lower_bound,upper_bound):
    age_alive,age_ok,age_note=verified_age_alive_mask(0,59)
    cohort,measured_mask,status_ok,status_note=verified_anthropometry_status_masks()
    if not age_ok or age_alive is None: return None,None,False,False,age_note
    if not status_ok or cohort is None or measured_mask is None: return None,None,False,False,status_note
    eligible=age_alive & cohort
    if variable not in data.columns: return None,eligible,True,False,f"{variable} is absent."
    if not label_matches(variable,label_pattern):
        return None,eligible,True,False,f"{variable} label mismatch; actual label: {variable_label(variable)}"
    score=pd.to_numeric(data[variable],errors="coerce")
    # DHS z-scores are stored at 100x; bounds are plausibility filters, not proof of validity.
    valid=score.between(lower_bound,upper_bound,inclusive="both")
    # Use indicator-specific score availability and plausibility. Do not require
    # hw13 == Measured for every index: WAZ needs weight and age, and valid WAZ
    # values can exist when the composite height/weight status is "Other".
    observed=eligible & valid
    target=pd.Series(np.nan,index=data.index,dtype="float64")
    target.loc[observed]=(score.loc[observed] < -200).astype(float)
    note=(f"{variable} label verified; score < -200 on DHS 100x scale; living age 0-59; "
          "provisional known-hw13 cohort excludes only 'no measurement found in household'; "
          f"target observed when the indicator-specific score is within [{lower_bound},{upper_bound}]. "
          "hw13 is audited as a measurement-status variable, not imposed as a universal measured-only gate: valid WAZ may exist with hw13='Other' because WAZ requires weight and age, not height. "
          "Out-of-range/special values and absent scores are missing outcomes. This cohort is not assumed to equal FR386 eligibility.")
    return target,eligible,True,True,note

def anthropometry_measurement_status_summary():
    living,ok,note=verified_age_alive_mask(0,59)
    if not ok or living is None:
        return pd.DataFrame([{"status_group":"not computable","n":None,"evidence":note}])
    if "hw13" not in data.columns or not label_matches("hw13",r"result of measurement.*height.*weight"):
        return pd.DataFrame([{"status_group":"not computable","n":None,"living_under5_reference_n":int(living.sum()),"evidence":"hw13 absent or label mismatch."}])
    labels=value_labels("hw13")
    measured=[c for c,l in labels.items() if re.fullmatch(r"\s*measured\s*",str(l),re.I)]
    no_measure=[c for c,l in labels.items() if re.search(r"no measurement found in household",str(l),re.I)]
    if len(measured)!=1:
        return pd.DataFrame([{"status_group":"not computable","n":None,"living_under5_reference_n":int(living.sum()),"evidence":"No unique measured value label."}])
    values=data["hw13"]; known=list(labels); ref=int(living.sum())
    masks=[
        ("measured",living & values.isin(measured)),
        ("no measurement found",living & values.isin(no_measure) if len(no_measure)==1 else pd.Series(False,index=data.index)),
        ("other recognized statuses pooled",living & values.notna() & values.isin(known) & ~values.isin(measured) & ~values.isin(no_measure)),
        ("missing/unclassified",living & (values.isna() | ~values.isin(known))),
    ]
    rows=[]
    for name,mask in masks:
        count=int(mask.sum())
        rows.append({"status_group":name,"n":count,"percent_of_living_under5":100*count/ref if ref else None,
                     "living_under5_reference_n":ref,"evidence":"Aggregate statuses; small categories pooled."})
    if len(no_measure)!=1:
        rows[1]["n"]=None; rows[1]["percent_of_living_under5"]=None
        rows[1]["evidence"]="No-measurement label is not unique; category not inferred."
    return pd.DataFrame(rows)

ANTHROPOMETRIC_SCORE_BOUNDS={"stunting":(-600,600),"underweight":(-600,500),"wasting":(-500,500)}
anthropometry_measurement_summary=anthropometry_measurement_status_summary()
anthropometry_measurement_summary.to_csv(OUTPUT_DIR/"anthropometry_measurement_status.csv",index=False)
display(anthropometry_measurement_summary)

def find_anemia_status_fields():
    matches = []
    for variable in data.columns:
        if not re.search(r"an[ae]mia", variable_label(variable), re.I):
            continue
        positive, negative = [], []
        for code, category in value_labels(variable).items():
            text = str(category).strip().lower()
            if re.search(r"\b(not|no)\b", text):
                negative.append(code)
            elif re.search(r"\b(an[ae]mia|an[ae]mic|mild|moderate|severe)\b", text):
                positive.append(code)
        if positive and negative:
            matches.append((variable, positive, negative))
    return matches

candidate_definitions = [
    {"candidate_outcome":"stunting","family":"anthropometric","required_variables":["hw70","hw13"],"direct":["hw70"],"derived":["hw2","hw3","hw70","hw71","hw72","hw73"],"construction":"Stunted if label-verified height-for-age z-score < -2 SD (score < -200); HAZ bounds [-600,+600] are plausibility limits.","eligibility":"Living age 0-59 in the provisional known-hw13 cohort excluding only 'no measurement found'; use valid HAZ and indicator-specific plausibility bounds. Reconcile with FR386."},
    {"candidate_outcome":"wasting","family":"anthropometric","required_variables":["hw72","hw13"],"direct":["hw72"],"derived":["hw2","hw3","hw70","hw71","hw72","hw73"],"construction":"Wasted if label-verified weight-for-height z-score < -2 SD (score < -200); WHZ bounds [-500,+500] are plausibility limits.","eligibility":"Living age 0-59 in the provisional known-hw13 cohort excluding only 'no measurement found'; use valid WHZ and indicator-specific plausibility bounds. Reconcile with FR386."},
    {"candidate_outcome":"underweight","family":"anthropometric","required_variables":["hw71","hw13"],"direct":["hw71"],"derived":["hw2","hw3","hw70","hw71","hw72","hw73"],"construction":"Underweight if label-verified weight-for-age z-score < -2 SD (score < -200); WAZ bounds [-600,+500] are plausibility limits.","eligibility":"Living age 0-59 in the provisional known-hw13 cohort excluding only 'no measurement found'; use valid WAZ and indicator-specific plausibility bounds; do not require height measurement. Reconcile with FR386."},
    {"candidate_outcome":"fever","family":"morbidity","required_variables":["h22"],"direct":["h22"],"derived":[],"construction":"Yes to verified fever question; response codes read from value labels, never hard-coded.","eligibility":"Provisional living children age 0-59 months; verify exact BDHS 2022 questionnaire universe and whether the item is asked for every eligible child before treating this as final."},
    {"candidate_outcome":"diarrhea","family":"morbidity","required_variables":["h11"],"direct":["h11"],"derived":[],"construction":"Two-week diarrhea indicator: combine every value-labeled affirmative category (including 'yes, last 24 hours' and 'yes, last two weeks') as Yes; retain the single labeled No category as No; don't know remains missing. Codes are read from value labels, never hard-coded.","eligibility":"Provisional living children age 0-59 months; verify exact BDHS 2022 questionnaire universe and whether the item is asked for every eligible child before treating this as final."},
    {"candidate_outcome":"ari_symptoms","family":"morbidity","required_variables":[],"direct":[],"derived":[],"construction":"TODO/REVIEW: verify BDHS 2022 report/questionnaire definition and all cough/breathing codes; no formula assumed.","eligibility":"Living children age 0-59 months; verify questionnaire universe."},
    {"candidate_outcome":"child_anemia","family":"biomarker","required_variables":[],"direct":[],"derived":[],"construction":"Compute only if metadata exposes one anemia-status field with labeled anemic/non-anemic categories; otherwise review measure, adjustment, cutoff, and standard.","eligibility":"Living children age 6-59 months with valid anemia status/measurement."},
    {"candidate_outcome":"full_age_appropriate_immunisation","family":"immunisation","required_variables":[],"direct":[],"derived":[],"construction":"TODO/REVIEW: verify national schedule, eligible age, doses, information source, and exact codes before creating a label.","eligibility":"Initial audit denominator 12-23 months; team must confirm this against BDHS 2022 documentation."},
    {"candidate_outcome":"mortality_related","family":"mortality","required_variables":["b5"],"direct":["b5"],"derived":[],"construction":"Descriptive death status only if b5 labels are verified. Not an ordinary binary target due differing exposure time/censoring.","eligibility":"KR birth records with verified alive status; time at risk must be addressed."},
]

candidate_rows, candidate_targets, candidate_eligibility = [], {}, {}
anemia_fields = find_anemia_status_fields()
for spec in candidate_definitions:
    name = spec["candidate_outcome"]
    target = eligible = None
    eligible_ok = constructed = False
    evidence = ""
    if name in {"stunting","wasting","underweight"}:
        variable = spec["required_variables"][0]
        pattern = {"stunting":r"height.*age|height/age","wasting":r"weight.*height|weight/height","underweight":r"weight.*age|weight/age"}[name]
        lower_bound,upper_bound=ANTHROPOMETRIC_SCORE_BOUNDS[name]
        target,eligible,eligible_ok,constructed,evidence=build_anthropometric_target(variable,pattern,lower_bound,upper_bound)
    elif name in {"fever","diarrhea"}:
        eligible, eligible_ok, note = verified_age_alive_mask(0, 59)
        target, constructed, evidence = build_binary_question(
            spec["required_variables"][0], r"fever" if name == "fever" else r"diarr?hoea|diarrhea",
            eligible, eligible_ok, note
        )
    elif name == "child_anemia":
        eligible, eligible_ok, note = verified_age_alive_mask(6, 59)
        if eligible_ok and len(anemia_fields) == 1:
            variable, positive_codes, negative_codes = anemia_fields[0]
            target = pd.Series(np.nan, index=data.index, dtype="float64")
            target.loc[eligible & data[variable].isin(positive_codes)] = 1
            target.loc[eligible & data[variable].isin(negative_codes)] = 0
            spec["required_variables"], spec["direct"] = [variable], [variable]
            constructed, evidence = True, f"Unique anemia-status field {variable} and labeled categories found; verify survey cutoffs."
        else:
            if len(anemia_fields) > 1:
                spec["required_variables"] = [item[0] for item in anemia_fields]
                spec["direct"] = [item[0] for item in anemia_fields]
            evidence = "No unique label-verified anemia status field/categories, or age eligibility is unverified. Raw hemoglobin requires human confirmation."
    elif name == "ari_symptoms":
        eligible, eligible_ok, evidence = verified_age_alive_mask(0, 59)
        fields = [str(v) for v in data.columns if re.search(r"cough|breath|chest", variable_label(v), re.I)]
        spec["required_variables"], spec["direct"] = fields, fields
        evidence = "Construction pending. Label-matched cough/breathing/chest fields: " + (", ".join(fields) if fields else "none")
    elif name == "full_age_appropriate_immunisation":
        eligible, eligible_ok, evidence = verified_age_alive_mask(12, 23)
        fields = [str(v) for v in data.columns if re.search(r"vaccin|immuni[sz]", variable_label(v), re.I)]
        spec["required_variables"], spec["direct"] = fields, fields
        evidence = "Construction pending. Label-matched vaccine fields: " + (", ".join(fields) if fields else "none")
    elif name == "mortality_related":
        if "b5" in data.columns and label_matches("b5", r"alive"):
            yes, no = yes_no_codes("b5")
            if len(yes) == 1 and len(no) == 1:
                eligible = data["b5"].isin(yes + no)
                eligible_ok = True
                target = pd.Series(np.nan, index=data.index, dtype="float64")
                target.loc[data["b5"].isin(no)] = 1
                target.loc[data["b5"].isin(yes)] = 0
                constructed = True
                evidence = "b5 label/value labels verified. Descriptive death status only; censoring/follow-up prohibits ordinary binary prediction."
            else:
                evidence = "b5 value labels do not uniquely identify alive and dead."
        else:
            evidence = "b5 absent or label does not confirm alive status."

    candidate_targets[name], candidate_eligibility[name] = target, eligible
    eligible_n = int(eligible.sum()) if eligible is not None else None
    if target is not None and eligible is not None:
        valid_target = target.loc[eligible].dropna()
        positive, negative, valid_n = int((valid_target == 1).sum()), int((valid_target == 0).sum()), int(len(valid_target))
        missing_n = int(eligible_n - valid_n)
        missing_pct = missing_n / eligible_n * 100 if eligible_n else np.nan
        prevalence = positive / valid_n if valid_n else np.nan
        imbalance = max(positive, negative) / min(positive, negative) if min(positive, negative) else np.nan
    else:
        positive = negative = valid_n = missing_n = None
        missing_pct = prevalence = imbalance = np.nan

    label_evidence = {
        variable: variable_label(variable) if variable in data.columns else "NOT PRESENT"
        for variable in spec["required_variables"]
    }
    candidate_rows.append({
        "candidate_outcome":name, "family":spec["family"],
        "status":"computed for audit" if constructed else "TODO / REVIEW — not computed",
        "required_variables":", ".join(spec["required_variables"]) or "definition-specific fields pending review",
        "actual_variable_labels":json.dumps(label_evidence, ensure_ascii=False),
        "construction_rule":spec["construction"], "eligibility_restriction":spec["eligibility"],
        "eligibility_evidence":evidence,
        "requires_derived_or_composite_definition":name in {"stunting","wasting","underweight","ari_symptoms","child_anemia","full_age_appropriate_immunisation"},
        "definition_actually_computed":bool(constructed),
        "total_available_observations":int(len(data)), "eligible_n":eligible_n,
        "valid_outcome_n":valid_n, "missing_outcome_n":missing_n,
        "missing_outcome_pct_of_eligible":missing_pct, "positive_n":positive, "negative_n":negative,
        "positive_prevalence":prevalence, "class_imbalance_ratio_larger_over_smaller":imbalance,
        "predictability_feasibility_eligible":bool(constructed and name != "mortality_related"),
    })

candidate_outcomes = pd.DataFrame(candidate_rows)
candidate_outcomes.to_csv(OUTPUT_DIR / "candidate_outcomes.csv", index=False)
display(candidate_outcomes[["candidate_outcome","status","eligible_n","valid_outcome_n","missing_outcome_pct_of_eligible","positive_n","negative_n","positive_prevalence","eligibility_evidence"]])
print("No outcome has been selected. Pending definitions require researcher review.")


### Anthropometry denominator bridge — file audit versus FR386 Table C.8

This comparison uses the private KR file's actual hw13 labels and published aggregate figures from the official report. It documents the discrepancy; it does not assume that the report determines the file's coding or explain the difference.

In [ ]:
FR386_TABLE_C8 = {
    "eligible_total":4420, "mother_interviewed":4199, "mother_not_interviewed_in_household":221,
    "valid_pct_interviewed":{"stunting":97.2,"wasting":97.2,"underweight":98.5},
    "source":"Bangladesh DHS 2022 Final Report, Table C.8",
    "url":"https://www.dhsprogram.com/pubs/pdf/FR386/FR386.pdf",
}
living, living_ok, living_note=verified_age_alive_mask(0,59)
status_cohort, measured_mask, hw13_ok, hw13_note=verified_anthropometry_status_masks()
if living_ok and living is not None and hw13_ok:
    living_n=int(living.sum())
    measured_n=int((living & measured_mask).sum())
    cohort_n=int((living & status_cohort).sum())
    no_measure_n=int((living & data["hw13"].notna() & ~status_cohort).sum())
    denominator_bridge=pd.DataFrame([
        {"quantity":"KR living children age 0-59, b5=Yes","n":living_n,"source":"Private KR file, labels verified","interpretation":"Reference universe in this recode."},
        {"quantity":"KR hw13 measured","n":measured_n,"source":"Private KR file, hw13 labels","interpretation":"Measured-status count; not full eligible denominator."},
        {"quantity":"KR provisional known-hw13 cohort excluding no-measurement-found","n":cohort_n,"source":"Private KR file, operational proxy","interpretation":"Candidate denominator; not asserted equal to FR386."},
        {"quantity":"KR hw13 no measurement found in household","n":no_measure_n,"source":"Private KR file, hw13 labels","interpretation":"Separate status group; excluded from provisional cohort."},
        {"quantity":"FR386 C.8 total eligible for anthropometry","n":FR386_TABLE_C8["eligible_total"],"source":FR386_TABLE_C8["source"],"interpretation":"Published denominator; source/recode scope may differ."},
        {"quantity":"FR386 C.8 mothers interviewed","n":FR386_TABLE_C8["mother_interviewed"],"source":FR386_TABLE_C8["source"],"interpretation":"Closest published subgroup for KR comparison; not presumed identical."},
        {"quantity":"FR386 C.8 mother not interviewed but in household","n":FR386_TABLE_C8["mother_not_interviewed_in_household"],"source":FR386_TABLE_C8["source"],"interpretation":"Included in published total; KR representation requires review."},
        {"quantity":"KR cohort minus FR386 interviewed-mother count","n":cohort_n-FR386_TABLE_C8["mother_interviewed"],"source":"Calculated aggregate difference","interpretation":"Unresolved; no cause assumed."},
        {"quantity":"FR386 total minus KR provisional cohort","n":FR386_TABLE_C8["eligible_total"]-cohort_n,"source":"Calculated aggregate difference","interpretation":"Unresolved; do not attribute solely to maternal interview status."},
    ])
else:
    denominator_bridge=pd.DataFrame([{"quantity":"Anthropometry denominator bridge","n":None,"source":"Not computable","interpretation":living_note if not living_ok else hw13_note}])
denominator_bridge.to_csv(OUTPUT_DIR/"anthropometry_denominator_bridge.csv",index=False)
validity_rows=[]
for name,report_pct in FR386_TABLE_C8["valid_pct_interviewed"].items():
    eligible=candidate_eligibility.get(name); target=candidate_targets.get(name)
    if eligible is not None and target is not None:
        n=int(eligible.sum()); valid_n=int((eligible & target.notna()).sum())
        observed_pct=100*valid_n/n if n else np.nan
        score_variable = {"stunting":"hw70","wasting":"hw72","underweight":"hw71"}[name]
        lower_bound, upper_bound = ANTHROPOMETRIC_SCORE_BOUNDS[name]
        score = pd.to_numeric(data[score_variable], errors="coerce")
        score_valid = score.between(lower_bound, upper_bound, inclusive="both")
        measured_mask_local = data["hw13"].isin([code for code,label in value_labels("hw13").items() if re.fullmatch(r"\s*measured\s*",str(label),re.I)])
        valid_measured = int((eligible & score_valid & measured_mask_local).sum())
        valid_other_status = int((eligible & score_valid & ~measured_mask_local).sum())
        validity_rows.append({"candidate_outcome":name,"kr_provisional_cohort_n":n,"kr_valid_score_n":valid_n,
          "kr_valid_score_n_hw13_measured":valid_measured,"kr_valid_score_n_hw13_other_status":valid_other_status,
          "kr_valid_pct_of_provisional_cohort":observed_pct,"fr386_c8_valid_pct_interviewed_mothers":report_pct,
          "difference_percentage_points":observed_pct-report_pct if pd.notna(observed_pct) else np.nan,
          "interpretation":"Rough aggregate comparison only; denominator/scope unresolved; indicator-specific valid scores govern. WAZ may be valid without the composite hw13 measured status."})
anthropometry_validity_comparison=pd.DataFrame(validity_rows)
anthropometry_validity_comparison.to_csv(OUTPUT_DIR/"anthropometry_validity_comparison.csv",index=False)


## 5 — Outcome quality assessment

Report denominators, outcome missingness, class counts, prevalence, and imbalance for outcomes built from verified metadata. Pending definitions retain an eligible-population count where possible, but outcome counts stay blank. There is no universal acceptable missingness or prevalence cutoff here. A class count smaller than the requested folds is a grouped-validation feasibility prompt, not an automatic exclusion.

In [ ]:
quality = candidate_outcomes.copy()
quality["quality_flags"] = quality.apply(
    lambda row: "; ".join(
        flag for flag, condition in [
            ("definition pending human review", row["status"] != "computed for audit"),
            ("no eligible observations", pd.notna(row["eligible_n"]) and row["eligible_n"] == 0),
            ("outcome missing for at least one eligible observation", pd.notna(row["missing_outcome_n"]) and row["missing_outcome_n"] > 0),
            ("single observed class", pd.notna(row["valid_outcome_n"]) and (row["positive_n"] == 0 or row["negative_n"] == 0)),
            (f"one class has fewer than {CV_FOLDS} observations; check cluster support", pd.notna(row["positive_n"]) and min(row["positive_n"], row["negative_n"]) < CV_FOLDS),
            ("mortality target has unequal follow-up/censoring; not a standard binary task", row["candidate_outcome"] == "mortality_related"),
        ] if condition
    ) or "No mechanical quality flag; human review still required.",
    axis=1,
)
quality.to_csv(OUTPUT_DIR / "outcome_quality.csv", index=False)
display(quality[["candidate_outcome","total_available_observations","eligible_n","valid_outcome_n","missing_outcome_n","missing_outcome_pct_of_eligible","positive_n","negative_n","positive_prevalence","class_imbalance_ratio_larger_over_smaller","quality_flags"]])

plot_data = quality.loc[quality["positive_prevalence"].notna()].copy()
if not plot_data.empty:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    axes[0].bar(plot_data["candidate_outcome"], plot_data["positive_prevalence"], color="#315A7D")
    axes[0].set_title("Observed positive prevalence")
    axes[0].set_ylabel("Proportion among valid eligible outcomes")
    axes[0].tick_params(axis="x", rotation=55)
    axes[0].grid(axis="y", alpha=0.2)
    missing_data = quality.loc[quality["missing_outcome_pct_of_eligible"].notna()]
    axes[1].bar(missing_data["candidate_outcome"], missing_data["missing_outcome_pct_of_eligible"], color="#B06B42")
    axes[1].set_title("Outcome missingness among eligible children")
    axes[1].set_ylabel("Percent")
    axes[1].tick_params(axis="x", rotation=55)
    axes[1].grid(axis="y", alpha=0.2)
    fig.tight_layout()
    plt.show()
    plt.close(fig)
else:
    print("No candidate has computed prevalence; review pending definitions.")

In [ ]:
quality.to_csv(OUTPUT_DIR/"outcome_quality.csv",index=False)
print("Absolute outcome quantities only; no candidate-relative median flags.")
display(quality[["candidate_outcome","eligible_n","valid_outcome_n","missing_outcome_n","missing_outcome_pct_of_eligible","positive_n","negative_n","positive_prevalence","class_imbalance_ratio_larger_over_smaller","quality_flags"]])


## 6 - Stage A: Leakage audit

For each candidate, inspect direct target fields, measurements used to derive the target, plausible post-outcome proxies, dictionary-screened pilot predictors, and survey identifiers. Correlation alone does not trigger removal. A feature is a proxy risk only when its label indicates a likely post-outcome treatment/care-seeking field or an outcome-domain response.

Pilot variables are discovered from actual dictionary labels using declared semantic domains, not a fixed list of variable names. Each variable remains subject to outcome-specific leakage checks and human review. These actions apply to the feasibility pilot only; they do not define the final thesis feature set.

In [ ]:
# Pilot predictor set - not final thesis feature set.
# Broad semantic domains and label patterns are predeclared before model results are examined.
PILOT_DOMAIN_RULES = [
    {"category":"Demographic","kind":"numeric","pattern":r"\bage of child\b|\bchild age\b|\bage.*months\b","reason":"Child-age label; timing and eligibility require review."},
    {"category":"Child characteristics","kind":"categorical","pattern":r"\bsex of child\b|\bchild.*sex\b","reason":"Child-sex label; review its role in anthropometric reference standards."},
    {"category":"Child characteristics","kind":"numeric","pattern":r"\bbirth order\b","reason":"Birth-order label; child characteristic domain."},
    {"category":"Maternal","kind":"numeric","pattern":r"\bage of respondent\b|\brespondent.*age\b","reason":"Respondent-age label; maternal demographic domain."},
    {"category":"Maternal","kind":"categorical","pattern":r"\beducation\b|\beducational level\b","reason":"Education label; verify whose education is recorded."},
    {"category":"Maternal","kind":"categorical","pattern":r"\bmarital status\b","reason":"Marital-status label; maternal characteristic domain."},
    {"category":"Maternal","kind":"categorical","pattern":r"\bcurrently working\b|\bworking status\b|\brespondent.*work","reason":"Work-status label; maternal characteristic domain."},
    {"category":"Socioeconomic","kind":"categorical","pattern":r"\bwealth index\b","reason":"Wealth-index label; review its construction."},
    {"category":"Geographic/survey-design","kind":"categorical","pattern":r"\btype of place of residence\b|\burban.*rural\b|\brural.*urban\b|\bregion\b|\bdivision\b","reason":"Residence or broad-region label; never use cluster/PSU identifiers as predictors."},
    {"category":"Household","kind":"numeric","pattern":r"\bnumber of household members\b|\bhousehold size\b","reason":"Household-size label; household context, not an identifier."},
    {"category":"Household","kind":"categorical","pattern":r"\bsource of drinking water\b|\bdrinking water source\b","reason":"Water-source label; household living-conditions domain."},
    {"category":"Household","kind":"categorical","pattern":r"\btoilet\b|\bsanitation\b","reason":"Toilet/sanitation label; household living-conditions domain."},
]
ADMIN_ID_SPECS = {
    "v001":("Geographic/survey-design","Cluster/PSU candidate; audit only, never a predictor."),
    "v021":("Geographic/survey-design","PSU candidate; audit only, never a predictor."),
    "v022":("Geographic/survey-design","Strata candidate; audit only, never a predictor."),
    "v023":("Geographic/survey-design","Strata candidate; audit only, never a predictor."),
    "v005":("Geographic/survey-design","Sampling weight; audit only in Notebook 0, never a predictor."),
    "v002":("Administrative/identifier","Household identifier component; never a predictor."),
    "v003":("Administrative/identifier","Mother/respondent line identifier component; never a predictor."),
    "caseid":("Administrative/identifier","Respondent identifier; never a predictor."),
    "b16":("Administrative/identifier","Child line identifier candidate; never a predictor."),
    "bidx":("Administrative/identifier","Birth column/index candidate; do not include automatically."),
}
def pilot_policy_match(variable):
    if variable in ADMIN_ID_SPECS: return None
    label = variable_label(variable)
    if not label or label=="NOT PRESENT": return None
    return next((rule for rule in PILOT_DOMAIN_RULES if re.search(rule["pattern"],label,re.I)),None)

PILOT_POLICY_CANDIDATES = {}
PILOT_AMBIGUOUS_POLICY_VARIABLES = set()
pilot_policy_rows=[]
variable_rule_matches = {}
for rule in PILOT_DOMAIN_RULES:
    matched=[str(v) for v in data.columns if re.search(rule["pattern"],variable_label(str(v)),re.I)]
    pilot_policy_rows.append({"category":rule["category"],"label_rule":rule["pattern"],
      "matched_variables":", ".join(matched) if matched else "none",
      "policy_result":"unique match; pilot candidate" if len(matched)==1 else ("ambiguous; human review" if matched else "no match; no substitution"),
      "final_thesis_feature":False})
    for variable in matched:
        variable_rule_matches.setdefault(variable, []).append(rule)

# A variable matching multiple concepts is held out instead of being silently assigned
# to whichever rule happened to appear last. Each predictor can enter only once.
PREDICTOR_SPECS=[]
for variable, rules in variable_rule_matches.items():
    if len(rules) != 1:
        PILOT_AMBIGUOUS_POLICY_VARIABLES.add(variable)
        continue
    rule=rules[0]
    item={**rule,"variable":variable,"label_pattern":rule["pattern"]}
    PILOT_POLICY_CANDIDATES[variable]=item
    PREDICTOR_SPECS.append(item)

pilot_predictor_policy=pd.DataFrame(pilot_policy_rows)
pilot_predictor_policy.to_csv(OUTPUT_DIR/"pilot_predictor_policy.csv",index=False)

def candidate_proxy_pattern(candidate):
    return {
        "stunting":r"height|weight|anthropometr|z.?score|growth",
        "wasting":r"height|weight|anthropometr|z.?score|growth",
        "underweight":r"height|weight|anthropometr|z.?score|growth",
        "fever":r"fever|treatment|medicine|advice|care.?seek|health.?facility|provider",
        "diarrhea":r"diarr?hoea|diarrhea|treatment|medicine|ors|zinc|advice|care.?seek|health.?facility|provider",
        "ari_symptoms":r"cough|breath|chest|treatment|medicine|advice|care.?seek|health.?facility|provider",
        "child_anemia":r"an[ae]mia|hemoglobin|haemoglobin",
        "full_age_appropriate_immunisation":r"vaccin|immuni[sz]",
        "mortality_related":r"alive|death|died|survival|age at death",
    }[candidate]
ANTHROPOMETRIC_DERIVED_LABEL_PATTERNS = {
    "hw2":r"weight","hw3":r"height|length","hw70":r"height.*age|height/age",
    "hw71":r"weight.*age|weight/age","hw72":r"weight.*height|weight/height","hw73":r"body mass index|bmi",
}
candidate_status = candidate_outcomes.set_index("candidate_outcome")["status"].to_dict()
leakage_rows = []
for spec in candidate_definitions:
    candidate = spec["candidate_outcome"]
    direct = set(spec.get("direct",[]))
    derived = set(spec.get("derived",[]))-direct
    relevant = direct|derived|set(PILOT_POLICY_CANDIDATES)|set(ADMIN_ID_SPECS)|set(spec.get("required_variables",[]))
    pattern = candidate_proxy_pattern(candidate)
    relevant.update(str(v) for v in data.columns if re.search(pattern,variable_label(v),re.I))
    for variable in sorted(relevant):
        present = variable in data.columns
        label = variable_label(variable) if present else "NOT PRESENT"
        computed = candidate_status.get(candidate)=="computed for audit"
        if variable in direct and computed and present:
            classification,reason,action = "direct_leakage","Label-verified target field/direct target component.","exclude from predictors"
        elif variable in direct:
            classification,reason,action = "uncertain","Target-related field is not part of a verified recipe.","human review; do not predict from it"
        elif variable in derived:
            expected = ANTHROPOMETRIC_DERIVED_LABEL_PATTERNS.get(variable)
            if computed and present and expected and label_matches(variable,expected):
                classification,reason,action = "derived_leakage","Actual label verifies a raw anthropometric input or z-score used in the target family.","exclude from predictors"
            else:
                classification,reason,action = "uncertain","Variable name alone does not verify meaning or leakage.","human review; exclude until label checked"
        elif variable in PILOT_POLICY_CANDIDATES:
            item = PILOT_POLICY_CANDIDATES[variable]
            if variable in PILOT_AMBIGUOUS_POLICY_VARIABLES:
                classification,reason,action="uncertain","Multiple dictionary variables match one pilot concept; none chosen automatically.","human review; hold from pilot"
            else:
                anthropometric_age_sex = candidate in {"stunting","wasting","underweight"} and bool(re.search(r"age|sex of child|child.*sex",label,re.I))
                if anthropometric_age_sex:
                    classification,reason,action = "uncertain","Age/sex may be available at prediction time but also define anthropometric reference/eligibility.","hold from pilot pending team decision; do not carry exclusion into final model automatically"
                elif re.search(pattern,label,re.I):
                    classification,reason,action = "proxy_risk","Label indicates outcome-domain or likely post-outcome treatment/care-seeking.","exclude pending timing review"
                else:
                    classification,reason,action = "legitimate_predictor",item["reason"],"pilot candidate only; prediction-time review required"
        elif variable in ADMIN_ID_SPECS:
            category,why = ADMIN_ID_SPECS[variable]
            role_spec = SURVEY_ROLE_SPECS.get(variable)
            verified = bool(present and role_spec and re.search(role_spec[1],label,re.I))
            classification,reason,action = "uncertain",f"{category}: {why} "+("Actual label supports this role." if verified else "Actual label is absent or does not verify this role."),"exclude from predictors; verify role from dictionary"
        elif variable in spec.get("required_variables",[]) and not computed:
            classification,reason,action = "uncertain","Field may be an unconfirmed outcome component.","human review; do not use as predictor"
        elif present and re.search(pattern,label,re.I):
            classification,reason,action = "proxy_risk","Same outcome domain or likely post-outcome treatment/care-seeking.","exclude pending timing review"
        else:
            classification,reason,action = "uncertain","Label alone does not establish safe prediction-time availability.","human review"
        leakage_rows.append({"candidate_outcome":candidate,"variable":variable,"classification":classification,"reason":reason,"action":action,"evidence":label,"present":bool(present)})
leakage_audit = pd.DataFrame(leakage_rows).drop_duplicates(["candidate_outcome","variable"],keep="last")
leakage_audit.to_csv(OUTPUT_DIR/"leakage_audit.csv",index=False)
leakage_summary = leakage_audit.groupby(["candidate_outcome","classification"],dropna=False).size().unstack(fill_value=0).reset_index()
display(leakage_summary)
print(f"Saved {len(leakage_audit):,} candidate-variable metadata assessments; no feature values were displayed.")


## 7 — Stage A: Survey structure

Report actual v001, v021, v022, and v023 labels/counts, exact v001/v021 value agreement, and whether their grouping partitions are identical. The team verifies BDHS documentation and confirms the PSU; the notebook never chooses automatically.

The mother key includes a cluster component. Cross-PSU nesting is structurally non-independent with that key and is reported as not independently estimable, not an observed zero. Confirm the identifier structure before grouped validation. No IDs are displayed. Weight and prediction estimand remain open decisions.


In [ ]:
def verified_role(variable):
    if variable not in data.columns: return False
    role_spec=SURVEY_ROLE_SPECS.get(variable)
    return bool(role_spec and re.search(role_spec[1],variable_label(variable),re.I))
verified_psu=[v for v in ["v001","v021"] if verified_role(v)]
def same_group_partition(left,right):
    if len(left)!=len(right) or not left.notna().equals(right.notna()): return False
    pairs=pd.DataFrame({"left":left,"right":right}).loc[left.notna()]
    if pairs.empty: return False
    return bool((pairs.groupby("left")["right"].nunique()<=1).all() and (pairs.groupby("right")["left"].nunique()<=1).all())
psu_values_identical=bool(len(verified_psu)==2 and data["v001"].equals(data["v021"]))
psu_partitions_identical=bool(len(verified_psu)==2 and same_group_partition(data["v001"],data["v021"]))
psu_values_differ=bool(len(verified_psu)==2 and not psu_values_identical)
verified_strata=[v for v in ["v022","v023"] if verified_role(v)]
strata_values_identical=bool(len(verified_strata)==2 and data["v022"].equals(data["v023"]))
strata_partitions_identical=bool(len(verified_strata)==2 and same_group_partition(data["v022"],data["v023"]))
verified_weight=verified_role("v005")
mother_key=["v001","v002","v003"] if all(verified_role(v) for v in ["v001","v002","v003"]) else None
child_id_verified=verified_role("b16")
caseid_verified=verified_role("caseid")
bidx_verified=verified_role("bidx")
birth_record_key=["caseid","bidx"] if caseid_verified and bidx_verified else None
if birth_record_key:
    key_complete=bool(data[birth_record_key].notna().all(axis=1).all())
    birth_key_duplicate_n=int(data.duplicated(birth_record_key,keep=False).sum())
    birth_record_key_unique=bool(key_complete and birth_key_duplicate_n==0)
else:
    key_complete=False; birth_key_duplicate_n=None; birth_record_key_unique=False
GROUP_VECTOR=None
GROUP_VARIABLE=TEAM_CONFIRMED_PSU_VARIABLE
GROUPING_LABEL=None
GROUPING_PROVISIONAL=False
if TEAM_CONFIRMED_PSU_VARIABLE is None and len(verified_psu)==2 and psu_partitions_identical:
    # Joint key avoids choosing one survey-design interpretation while preserving disjoint folds under both.
    pair_data=data[["v001","v021"]]
    codes,_=pd.factorize(pd.MultiIndex.from_frame(pair_data),sort=False)
    GROUP_VECTOR=pd.Series(codes,index=data.index,dtype="Int64")
    GROUP_VECTOR.loc[pair_data.isna().any(axis=1)]=pd.NA
    GROUPING_LABEL="joint v001+v021 equivalence partition"
    GROUPING_PROVISIONAL=True
    group_resolution="Provisional grouped feasibility: v001/v021 labels are verified and partitions identical. Joint partition avoids choosing a canonical PSU; human confirmation of survey-design meaning is still required before final interpretation."
elif TEAM_CONFIRMED_PSU_VARIABLE is not None and TEAM_CONFIRMED_PSU_VARIABLE in verified_psu:
    GROUP_VECTOR=data[TEAM_CONFIRMED_PSU_VARIABLE].copy()
    GROUPING_LABEL=f"team-confirmed {TEAM_CONFIRMED_PSU_VARIABLE}"
    group_resolution=f"Team-confirmed grouping variable: {TEAM_CONFIRMED_PSU_VARIABLE}."
else:
    group_resolution="Grouped feasibility blocked: verify both PSU candidates have identical partitions or confirm a label-verified PSU from BDHS documentation."
role_index=survey_role_inventory.set_index("variable")
survey_variable_details={}
for variable in ["v001","v021","v022","v023"]:
    if variable in role_index.index:
        rec=role_index.loc[variable]
        survey_variable_details[variable]={"label":str(rec["actual_variable_label"]),"role_under_review":str(rec["role_under_review"]),"label_check":str(rec["label_check"]),"distinct_nonmissing_values":int(rec["unique_nonmissing_values"]) if pd.notna(rec["unique_nonmissing_values"]) else None}
    else:
        survey_variable_details[variable]={"label":"NOT PRESENT","role_under_review":"not verified","label_check":"not present","distinct_nonmissing_values":None}
psu_candidate_summaries={}
for psu in verified_psu:
    sizes=data.groupby(psu,dropna=True,sort=False).size()
    psu_candidate_summaries[psu]={"cluster_count":int(sizes.size),"children_per_cluster_median":float(sizes.median()) if len(sizes) else None,"children_per_cluster_max":int(sizes.max()) if len(sizes) else None}
structural_mother_nesting=bool(mother_key and GROUP_VECTOR is not None and psu_partitions_identical and "v001" in mother_key)
mother_span_results={}
if structural_mother_nesting:
    for psu in verified_psu:
        mother_span_results[psu]={"status":"structurally nested: verified mother key contains v001 and PSU partitions are identical; cross-PSU count would be tautological","mothers_spanning_multiple_psus_n":None,"mothers_with_psu_observed_n":None,"mothers_spanning_multiple_psus_pct":None}
elif mother_key and GROUP_VECTOR is not None:
    tmp=data[mother_key].copy(); tmp["_group"]=GROUP_VECTOR
    complete=tmp.notna().all(axis=1)
    spans=tmp.loc[complete].groupby(mother_key,sort=False)["_group"].nunique(dropna=True)
    denom=int(len(spans)); cross=int((spans>1).sum())
    mother_span_results["pilot_partition"]={"status":"computed from verified mother key and pilot partition","mothers_spanning_multiple_psus_n":cross,"mothers_with_psu_observed_n":denom,"mothers_spanning_multiple_psus_pct":float(100*cross/denom) if denom else None}
else:
    mother_span_results["all_candidates"]={"status":"not estimable: verified composite mother key or grouping partition unavailable","mothers_spanning_multiple_psus_n":None,"mothers_with_psu_observed_n":None,"mothers_spanning_multiple_psus_pct":None}
# Do not let all([]) accidentally pass the gate when the mother key is unavailable.
observed_mother_span_checks = [
    x.get("mothers_spanning_multiple_psus_n")
    for x in mother_span_results.values()
    if x.get("mothers_spanning_multiple_psus_n") is not None
]
if structural_mother_nesting:
    GROUP_VALIDATION_READY = GROUP_VECTOR is not None and mother_key is not None and birth_record_key_unique
elif GROUP_VECTOR is not None and mother_key is not None and observed_mother_span_checks:
    GROUP_VALIDATION_READY = all(value == 0 for value in observed_mother_span_checks) and birth_record_key_unique
else:
    GROUP_VALIDATION_READY = False
group_validation_gate=group_resolution if GROUP_VALIDATION_READY else group_resolution+" Mother nesting and/or unique child-record key could not be verified."
survey_structure={
 "psu_strata_candidate_labels_and_distinct_counts":survey_variable_details,"psu_candidate_aggregate_structure":psu_candidate_summaries,
 "verified_psu_candidates":verified_psu,"v001_v021_values_identical_if_both_label_verified":psu_values_identical if len(verified_psu)==2 else None,
 "v001_v021_values_differ_if_both_label_verified":psu_values_differ if len(verified_psu)==2 else None,
 "v001_v021_partitions_identical_if_both_label_verified":psu_partitions_identical if len(verified_psu)==2 else None,
 "team_confirmed_psu_variable":TEAM_CONFIRMED_PSU_VARIABLE,"group_variable_used_for_pilot":GROUP_VARIABLE,
 "grouping_partition_used_for_pilot":GROUPING_LABEL,"grouping_provisional_pending_survey_documentation":GROUPING_PROVISIONAL,
 "group_resolution":group_resolution,"group_validation_ready":GROUP_VALIDATION_READY,"group_validation_gate":group_validation_gate,
 "verified_strata_variables":verified_strata,
 "v022_v023_values_identical_if_both_label_verified":strata_values_identical if len(verified_strata)==2 else None,
 "v022_v023_partitions_identical_if_both_label_verified":strata_partitions_identical if len(verified_strata)==2 else None,
 "verified_sampling_weight_variable":"v005" if verified_weight else None,
 "verified_mother_key_components":mother_key or [],"verified_child_line_identifier":"b16" if child_id_verified else None,
 "verified_birth_record_key_components":birth_record_key or [],"birth_record_key_complete":key_complete,
 "birth_record_key_duplicate_rows":birth_key_duplicate_n,"birth_record_key_unique":birth_record_key_unique,
 "mother_to_psu_nesting_by_candidate":mother_span_results,"mother_nesting_structurally_guaranteed_by_composite_key":structural_mother_nesting,
 "cluster_count":int(GROUP_VECTOR.nunique(dropna=True)) if GROUP_VECTOR is not None else None,
 "children_per_cluster_median":float(GROUP_VECTOR.value_counts().median()) if GROUP_VECTOR is not None and not GROUP_VECTOR.dropna().empty else None,
 "children_per_cluster_max":int(GROUP_VECTOR.value_counts().max()) if GROUP_VECTOR is not None and not GROUP_VECTOR.dropna().empty else None,
 "mother_count":None,"children_per_mother_median":None,"children_per_mother_max":None,
}
if mother_key:
    complete_mothers=data[mother_key].notna().all(axis=1)
    mother_sizes=data.loc[complete_mothers].groupby(mother_key,dropna=True,sort=False).size()
    survey_structure.update({"mother_count":int(mother_sizes.size),"children_per_mother_median":float(mother_sizes.median()) if len(mother_sizes) else None,"children_per_mother_max":int(mother_sizes.max()) if len(mother_sizes) else None})
cluster_support_rows=[]
for candidate in candidate_outcomes["candidate_outcome"]:
    target,eligible=candidate_targets.get(candidate),candidate_eligibility.get(candidate)
    for psu in verified_psu or [None]:
        if psu is None:
            cluster_support_rows.append({"candidate_outcome":candidate,"psu_candidate":None,"team_confirmed_choice":False,"cluster_count":None,"positive_clusters":None,"negative_clusters":None,"status":"not assessed: no label-verified PSU candidate"})
        elif target is None or not isinstance(eligible,pd.Series):
            cluster_support_rows.append({"candidate_outcome":candidate,"psu_candidate":psu,"team_confirmed_choice":psu==GROUP_VARIABLE,"cluster_count":None,"positive_clusters":None,"negative_clusters":None,"status":"not assessed: outcome construction pending"})
        else:
            observed=eligible & target.notna() & data[psu].notna()
            frame=pd.DataFrame({"cluster":data.loc[observed,psu],"outcome":target.loc[observed].astype(int)})
            cluster_support_rows.append({"candidate_outcome":candidate,"psu_candidate":psu,"team_confirmed_choice":psu==GROUP_VARIABLE,"cluster_count":int(frame.cluster.nunique()),"positive_clusters":int(frame.loc[frame.outcome==1,"cluster"].nunique()),"negative_clusters":int(frame.loc[frame.outcome==0,"cluster"].nunique()),"status":"computed from outcome only"})
cluster_support=pd.DataFrame(cluster_support_rows)
cluster_support.to_csv(OUTPUT_DIR/"cluster_support_by_outcome.csv",index=False)
(OUTPUT_DIR/"survey_structure.json").write_text(json.dumps(survey_structure,indent=2,ensure_ascii=False,default=str),encoding="utf-8")
print(json.dumps(survey_structure,indent=2,ensure_ascii=False,default=str))
print("Candidate cluster support uses outcomes only; equivalence grouping is provisional pending human documentation review.")


## 8 — Stage B: Preliminary prediction-feasibility feature set

This compact, researcher-designed pilot policy matches semantic concepts to actual dictionary labels before model results. Only unique matches can enter; ambiguous matches remain pending. It is not neutral, exhaustive, or the final thesis feature set, and its coverage can affect feasibility. Prediction time, age/sex use, leakage, and final predictors require human review.


In [ ]:
leakage_classes_by_candidate = {
    candidate: set(leakage_audit.loc[
        (leakage_audit["candidate_outcome"] == candidate)
        & leakage_audit["classification"].isin(["direct_leakage","derived_leakage","proxy_risk"]),
        "variable"
    ])
    for candidate in candidate_outcomes["candidate_outcome"]
}
FEATURES_BY_OUTCOME = {}
predictor_rows = []
for candidate in candidate_outcomes["candidate_outcome"]:
    features = []
    for item in PREDICTOR_SPECS:
        variable = item["variable"]
        present = variable in data.columns
        label = variable_label(variable) if present else "NOT PRESENT"
        verified = present and bool(re.search(item["label_pattern"], label, re.I))
        match = leakage_audit.loc[
            (leakage_audit["candidate_outcome"] == candidate) & (leakage_audit["variable"] == variable),
            "classification"
        ]
        leak_class = match.iloc[0] if len(match) else "uncertain"
        if leak_class != "legitimate_predictor":
            disposition, reason = "exclude pending review", f"Leakage/prediction-time audit classification: {leak_class}."
        elif not verified:
            disposition, reason = "exclude pending review", "Absent or actual label does not match expected meaning."
        else:
            disposition, reason = "preliminary candidate", item["reason"]
            features.append(item)
        predictor_rows.append({
            "candidate_outcome":candidate,"variable":variable,"predictor_category":item["category"],
            "declared_type":item["kind"],"actual_variable_label":label,"present":bool(present),
            "label_verified":bool(verified),"leakage_classification":leak_class,
            "disposition":disposition,"reason_for_disposition":reason,
        })
    for variable, (category, reason) in ADMIN_ID_SPECS.items():
        predictor_rows.append({
            "candidate_outcome":candidate,"variable":variable,"predictor_category":category,
            "declared_type":"administrative / survey field",
            "actual_variable_label":variable_label(variable) if variable in data.columns else "NOT PRESENT",
            "present":bool(variable in data.columns),"label_verified":bool(verified_role(variable)),
            "leakage_classification":"uncertain","disposition":"exclude","reason_for_disposition":reason,
        })
    FEATURES_BY_OUTCOME[candidate] = features
predictor_inventory = pd.DataFrame(predictor_rows).drop_duplicates(["candidate_outcome","variable"], keep="last")
predictor_inventory.to_csv(OUTPUT_DIR / "predictor_inventory.csv", index=False)
display(predictor_inventory.groupby(["candidate_outcome","predictor_category","disposition"], dropna=False).size().rename("variable_count").reset_index())
print("Unique dictionary-label matches passing candidate review are pilot candidates only; none are final thesis features.")


In [ ]:
extra_predictor_rows = []
already_listed = set(zip(predictor_inventory["candidate_outcome"], predictor_inventory["variable"]))
for _, item in leakage_audit.iterrows():
    key = (item["candidate_outcome"], item["variable"])
    if key in already_listed:
        continue
    cls = item["classification"]
    category = "Potential leakage" if cls in {"direct_leakage","derived_leakage"} else ("Health/behavioral" if cls == "proxy_risk" else "Uncertain")
    variable = item["variable"]
    extra_predictor_rows.append({
        "candidate_outcome":item["candidate_outcome"],
        "variable":variable,
        "predictor_category":category,
        "declared_type":"target-domain, proxy, or unresolved field",
        "actual_variable_label":item["evidence"],
        "present":bool(item["present"]),
        "label_verified":cls in {"direct_leakage","derived_leakage","proxy_risk"},
        "leakage_classification":cls,
        "disposition":item["action"],
        "reason_for_disposition":item["reason"],
    })
if extra_predictor_rows:
    predictor_inventory = pd.concat([predictor_inventory, pd.DataFrame(extra_predictor_rows)], ignore_index=True)
predictor_inventory.to_csv(OUTPUT_DIR / "predictor_inventory.csv", index=False)
display(predictor_inventory.groupby(["candidate_outcome","predictor_category","disposition"],dropna=False).size().rename("variable_count").reset_index())
print("Inventory now includes potential leakage, health/behavioral proxy, and uncertain categories.")

## 9 — Stage B: Predictive Feasibility Check

This is preliminary, unweighted, cluster-held-out screening with Logistic Regression and Random Forest. It asks whether a task merits further feasibility study; it does not rank or select outcomes. It is blocked until both researchers confirm the outcome definition, candidate-specific leakage audit, prediction/measurement-time boundary, target estimand, and intentional unweighted-pilot decision in cell 3. The team must inspect the PSU and mother-key relation. If the notebook uses the joint v001/v021 equivalence partition before a canonical PSU is confirmed, every resulting fold metric is provisional and cannot be treated as final performance evidence. Preprocessing is fitted within each training fold. Report fold ROC-AUC and average precision (AP); AP is not trapezoidal PR-AUC. Each test fold's prevalence is its AP no-skill reference. Fold means are descriptive, not precise population estimates.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score
import time

def make_one_hot_encoder():
    try: return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError: return OneHotEncoder(handle_unknown="ignore", sparse=True)

def make_preprocessor(feature_specs):
    numeric = [x["variable"] for x in feature_specs if x["kind"] == "numeric"]
    categorical = [x["variable"] for x in feature_specs if x["kind"] == "categorical"]
    transformers = []
    if numeric:
        transformers.append(("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("scale", StandardScaler()),
        ]), numeric))
    if categorical:
        transformers.append(("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__", keep_empty_features=True)),
            ("onehot", make_one_hot_encoder()),
        ]), categorical))
    return ColumnTransformer(transformers=transformers, remainder="drop")

signal_fold_rows=[]
signal_fold_audit_rows=[]
def run_grouped_signal_check(candidate):
    global signal_fold_rows, signal_fold_audit_rows
    if not RUN_PREDICTIVE_FEASIBILITY_SCREEN:
        return [], "Predictive feasibility screen disabled by team configuration."
    if candidate not in TEAM_REVIEWED_OUTCOME_DEFINITIONS or candidate not in TEAM_CONFIRMED_OUTCOME_DEFINITIONS:
        return [], "Outcome definition has not been reviewed and confirmed by both researchers in cell 3."
    if candidate not in TEAM_REVIEWED_LEAKAGE_AUDITS or candidate not in TEAM_CONFIRMED_LEAKAGE_AUDITS:
        return [], "Candidate-specific leakage audit has not been reviewed and confirmed by both researchers in cell 3."
    if (not TEAM_CONFIRMED_PREDICTION_TIME
            or not PREDICTION_TIME_DEFINITION.strip()
            or PREDICTION_TIME_DEFINITION.strip().upper().startswith("OPEN")):
        return [], "Prediction/measurement-time boundary is still open or not confirmed in cell 3."
    if (not TEAM_CONFIRMED_PREDICTION_POPULATION
            or not PREDICTION_POPULATION_DECISION.strip()
            or PREDICTION_POPULATION_DECISION.strip().upper().startswith("OPEN")):
        return [], "Target population/estimand decision is still open or not confirmed in cell 3."
    if (not TEAM_CONFIRMED_UNWEIGHTED_PILOT
            or SURVEY_WEIGHT_MODELING_DECISION.strip().upper().startswith("OPEN")
            or "unweighted" not in SURVEY_WEIGHT_MODELING_DECISION.lower()):
        return [], "The intentionally unweighted pilot-screen decision is not explicitly confirmed in cell 3."
    target = candidate_targets.get(candidate)
    eligible = candidate_eligibility.get(candidate)
    feature_specs = FEATURES_BY_OUTCOME.get(candidate, [])
    if target is None or eligible is None:
        return [], "Outcome has not been constructed from verified metadata."
    if GROUP_VECTOR is None or not GROUP_VALIDATION_READY:
        return [], group_validation_gate
    if not feature_specs:
        return [], "No compact predictor passed metadata and leakage checks."
    observed = eligible & target.notna()
    X = data.loc[observed, [x["variable"] for x in feature_specs]].copy()
    for item in feature_specs:
        if item["kind"] == "categorical":
            # Stata categorical predictors are often numeric codes. Convert every
            # nonmissing category to text before inserting a text missing token;
            # otherwise OneHotEncoder receives mixed int/str values and can fail.
            X[item["variable"]] = X[item["variable"]].map(
                lambda value: "__MISSING__" if pd.isna(value) else str(value)
            ).astype(str)
    y = target.loc[observed].astype(int)
    groups = GROUP_VECTOR.loc[observed]
    keep = groups.notna()
    X, y, groups = X.loc[keep], y.loc[keep], groups.loc[keep]
    if y.nunique() != 2:
        return [], "Only one outcome class remains after eligibility and valid-group filtering."
    group_table = pd.DataFrame({"group":groups, "y":y})
    positive_groups = int(group_table.loc[group_table["y"] == 1, "group"].nunique())
    negative_groups = int(group_table.loc[group_table["y"] == 0, "group"].nunique())
    n_splits = min(CV_FOLDS, int(groups.nunique()), positive_groups, negative_groups)
    if n_splits < 2:
        return [], "Fewer than two distinct clusters contain each outcome class; grouped CV is not feasible."
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
    try:
        splits = list(splitter.split(X, y, groups))
    except Exception as exc:
        return [], f"StratifiedGroupKFold could not construct splits ({type(exc).__name__}: {str(exc)[:240]})."
    factories = {
        "Logistic Regression":lambda: LogisticRegression(max_iter=500, random_state=RANDOM_SEED),
        "Random Forest":lambda: RandomForestClassifier(n_estimators=120, min_samples_leaf=3, n_jobs=1, random_state=RANDOM_SEED),
    }
    rows = []
    for model_name, factory in factories.items():
        model_started=time.perf_counter()
        aucs, aps, ap_baselines, error_type = [], [], [], None
        for fold_idx, (train_idx, test_idx) in enumerate(splits, start=1):
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
            train_groups = set(groups.iloc[train_idx].tolist())
            test_groups = set(groups.iloc[test_idx].tolist())
            if train_groups.intersection(test_groups):
                raise RuntimeError("Group leakage detected between train and test.")
            if y_test.nunique() != 2 or y_train.nunique() != 2:
                reason = "single_class_test_fold" if y_test.nunique() != 2 else "single_class_training_fold"
                signal_fold_audit_rows.append({"candidate_outcome":candidate,"model":model_name,"fold":fold_idx,
                    "fold_status":"skipped","skip_reason":reason,"train_n":int(len(y_train)),"test_n":int(len(y_test)),
                    "test_positive_n":int(y_test.sum()),"test_negative_n":int(len(y_test)-y_test.sum()),
                    "group_variable":GROUPING_LABEL,"grouping_provisional":GROUPING_PROVISIONAL})
                continue
            pipeline = Pipeline([
                ("preprocess", make_preprocessor(feature_specs)),
                ("model", factory()),
            ])
            try:
                fold_started=time.perf_counter()
                pipeline.fit(X.iloc[train_idx], y_train)
                probabilities = pipeline.predict_proba(X.iloc[test_idx])[:, 1]
                aucs.append(float(roc_auc_score(y_test, probabilities)))
                aps.append(float(average_precision_score(y_test, probabilities)))
                ap_baselines.append(float(y_test.mean()))
                fold_number = fold_idx
                signal_fold_rows.append({"candidate_outcome":candidate,"model":model_name,"fold":fold_number,
                    "roc_auc":aucs[-1],"average_precision":aps[-1],
                    "ap_no_skill_reference_test_prevalence":ap_baselines[-1],
                    "test_n":int(len(y_test)),"test_positive_n":int(y_test.sum()),
                    "test_negative_n":int(len(y_test)-y_test.sum()),
                    "group_variable":GROUPING_LABEL,"grouping_provisional":GROUPING_PROVISIONAL,"grouping_design_confirmation_pending":GROUPING_PROVISIONAL,"fold_runtime_seconds":float(time.perf_counter()-fold_started),"train_test_group_overlap":False})
                signal_fold_audit_rows.append({"candidate_outcome":candidate,"model":model_name,"fold":fold_idx,
                    "fold_status":"completed","skip_reason":None,"train_n":int(len(y_train)),"test_n":int(len(y_test)),
                    "test_positive_n":int(y_test.sum()),"test_negative_n":int(len(y_test)-y_test.sum()),
                    "group_variable":GROUPING_LABEL,"grouping_provisional":GROUPING_PROVISIONAL})
            except Exception as exc:
                error_type = f"{type(exc).__name__}: {str(exc)[:240]}"
                signal_fold_audit_rows.append({"candidate_outcome":candidate,"model":model_name,"fold":fold_idx,
                    "fold_status":"error","skip_reason":error_type,"train_n":int(len(y_train)),"test_n":int(len(y_test)),
                    "test_positive_n":int(y_test.sum()),"test_negative_n":int(len(y_test)-y_test.sum()),
                    "group_variable":GROUPING_LABEL,"grouping_provisional":GROUPING_PROVISIONAL})
                break
        rows.append({
            "candidate_outcome":candidate,"model":model_name,
            "roc_auc_mean":float(np.mean(aucs)) if aucs else np.nan,
            "roc_auc_sd":float(np.std(aucs, ddof=1)) if len(aucs)>1 else (0.0 if aucs else np.nan),
            "average_precision_mean":float(np.mean(aps)) if aps else np.nan,
            "average_precision_sd":float(np.std(aps, ddof=1)) if len(aps)>1 else (0.0 if aps else np.nan),
            "folds_completed":int(min(len(aucs),len(aps))),"folds_requested_for_candidate":int(n_splits),
            "positive_clusters":positive_groups,"negative_clusters":negative_groups,
            "eligible_valid_n_used":int(len(y)),"prevalence_baseline_for_average_precision":float(np.mean(ap_baselines)) if ap_baselines else np.nan,
            "group_variable":GROUPING_LABEL,"grouping_provisional":GROUPING_PROVISIONAL,"grouping_design_confirmation_pending":GROUPING_PROVISIONAL,"runtime_seconds":float(time.perf_counter()-model_started),
            "status":("failed" if not aucs else "partial") if error_type else ("completed" if len(aucs)==len(splits) else ("partial" if aucs else "failed")),
            "error_type":error_type,
        })
    return rows, None

signal_rows, signal_skips = [], []
for candidate in candidate_outcomes.loc[candidate_outcomes["predictability_feasibility_eligible"], "candidate_outcome"]:
    rows, reason = run_grouped_signal_check(candidate)
    signal_rows.extend(rows)
    if reason: signal_skips.append({"candidate_outcome":candidate,"reason":reason})
signal_check = pd.DataFrame(signal_rows)
if signal_check.empty:
    signal_check = pd.DataFrame(columns=[
        "candidate_outcome","model","roc_auc_mean","roc_auc_sd",
        "average_precision_mean","average_precision_sd",
        "folds_completed","folds_requested_for_candidate","positive_clusters",
        "negative_clusters","eligible_valid_n_used","prevalence_baseline_for_average_precision",
        "group_variable","grouping_provisional","grouping_design_confirmation_pending","runtime_seconds","status","error_type"
    ])
signal_check.to_csv(OUTPUT_DIR / "signal_check.csv", index=False)
signal_fold_metrics=pd.DataFrame(signal_fold_rows)
signal_fold_metrics.to_csv(OUTPUT_DIR/"signal_check_fold_metrics.csv",index=False)
signal_fold_audit=pd.DataFrame(signal_fold_audit_rows, columns=[
    "candidate_outcome","model","fold","fold_status","skip_reason","train_n","test_n",
    "test_positive_n","test_negative_n","group_variable","grouping_provisional"
])
signal_fold_audit.to_csv(OUTPUT_DIR/"signal_check_fold_audit.csv",index=False)
pd.DataFrame(signal_skips, columns=["candidate_outcome","reason"]).to_csv(OUTPUT_DIR / "signal_check_skips.csv", index=False)
display(signal_check)
if not signal_check.empty:
    chart_data = signal_check.loc[signal_check["roc_auc_mean"].notna()]
    if not chart_data.empty:
        fig, ax = plt.subplots(figsize=(10,5))
        for name, subset in chart_data.groupby("model"):
            ax.plot(subset["candidate_outcome"], subset["roc_auc_mean"], marker="o", label=name)
        ax.axhline(0.5, linestyle="--", color="gray", linewidth=1, label="ROC-AUC no-skill reference")
        ax.set_ylim(0,1)
        ax.set_ylabel("Mean ROC-AUC across valid cluster-held-out folds")
        ax.set_title("Predictive feasibility — descriptive, not an outcome-ranking rule")
        ax.tick_params(axis="x", rotation=45)
        ax.grid(axis="y", alpha=0.2)
        ax.legend()
        fig.tight_layout()
        plt.show()
        plt.close(fig)
print("Preprocessing is fitted inside training folds. No naive random split is used. Check signal_check_fold_audit.csv: completed means all requested folds succeeded; partial means at least one fold was skipped/failed; failed means no valid fold completed. Joint v001/v021 equivalence-partition metrics remain provisional until the team confirms PSU meaning from BDHS documentation.")


## 10 — Topic evidence matrix

Report raw counts and qualitative evidence only. There are no 1–5 scores, weighted totals, candidate-relative median flags, or automatic outcome recommendations. Predictability is a feasibility check, not an outcome selector.


In [ ]:
def leakage_counts(candidate):
    rows=leakage_audit.loc[leakage_audit.candidate_outcome==candidate]
    counts=rows.classification.value_counts().to_dict()
    return "; ".join(f"{k}={int(counts.get(k,0))}" for k in ["direct_leakage","derived_leakage","proxy_risk","legitimate_predictor","uncertain"])

def cluster_summary(candidate):
    rows=cluster_support.loc[cluster_support.candidate_outcome==candidate]
    parts=[f"{r.psu_candidate}: clusters={int(r.cluster_count)}, positive={int(r.positive_clusters)}, negative={int(r.negative_clusters)}"
           for _,r in rows.iterrows() if pd.notna(r.cluster_count)]
    return " | ".join(parts) if parts else "Not assessed; PSU/outcome pending."

matrix=[]
for _,r in candidate_outcomes.iterrows():
    c=r.candidate_outcome
    pred=predictor_inventory.loc[predictor_inventory.candidate_outcome==c]
    selected=int((pred.disposition=="preliminary candidate").sum())
    pending=int((pred.disposition=="exclude pending review").sum())
    metrics=signal_check.loc[signal_check.candidate_outcome==c]
    skipped=next((x["reason"] for x in signal_skips if x["candidate_outcome"]==c),"Not run.")
    feas=("Provisional grouped screen; PSU documentation confirmation pending; see signal_check.csv and signal_check_fold_metrics.csv." if GROUPING_PROVISIONAL else "Grouped descriptive screen completed; see signal_check.csv and signal_check_fold_metrics.csv.") if not metrics.empty else "Not completed: "+skipped
    runtime=f"Pilot runtime {metrics.runtime_seconds.sum():.1f} seconds; repeat in target Colab." if not metrics.empty else "Not benchmarked."
    matrix.append({
        "candidate_outcome":c,"label_validity":"Metadata-checked; human definition review remains." if r.status=="computed for audit" else "Pending; no outcome constructed.",
        "eligible_n":r.eligible_n,"valid_n":r.valid_outcome_n,"missing_n":r.missing_outcome_n,
        "missing_pct":r.missing_outcome_pct_of_eligible,"positive_n":r.positive_n,"negative_n":r.negative_n,
        "prevalence":r.positive_prevalence,"imbalance_ratio":r.class_imbalance_ratio_larger_over_smaller,
        "leakage_evidence":leakage_counts(c)+"; see leakage_audit.csv.",
        "cluster_support":cluster_summary(c),"predictability_feasibility":feas,
        "predictor_interpretability":f"{selected} pilot fields passed; {pending} held for review; not final features.",
        "literature_relevance":"Pending full-paper review; G2 remains provisional.",
        "colab_feasibility":runtime,"team_decision":"NOT SELECTED — human team decision required.",
    })
topic_evidence_matrix=pd.DataFrame(matrix)
topic_evidence_matrix.to_csv(OUTPUT_DIR/"topic_evidence_matrix.csv",index=False)
display(topic_evidence_matrix)
print("Evidence matrix saved; no numerical score or automatic outcome choice is calculated.")


## 11 — Reproducibility manifest

The run ID identifies an execution. The configuration identity incorporates the dataset hash, notebook version, Git commit when available, and configuration. Only aggregate metadata are saved; the raw file is not copied.


In [ ]:
manifest = {
    "run_id":RUN_ID,"run_timestamp_utc":RUN_TIMESTAMP_UTC,
    "experiment_config_id":EXPERIMENT_CONFIG_ID,"experiment_config_identity_sha256":CONFIGURATION_IDENTITY_SHA256,
    "dataset_sha256":DATASET_SHA256,"dataset_file_size_bytes":int(FILE_SIZE_BYTES),
    "reference_input_files":REFERENCE_INPUT_FILES,
    "dataset_metadata_file":"dataset_metadata.json",
    "output_files":["dataset_metadata.json","variable_inventory.csv","candidate_outcomes.csv","outcome_quality.csv","anthropometry_measurement_status.csv","anthropometry_denominator_bridge.csv","anthropometry_validity_comparison.csv","leakage_audit.csv","pilot_predictor_policy.csv","predictor_inventory.csv","survey_variable_role_checks.csv","cluster_support_by_outcome.csv","survey_structure.json","signal_check.csv","signal_check_fold_metrics.csv","signal_check_fold_audit.csv","signal_check_skips.csv","topic_evidence_matrix.csv","reference_source_audit.json","variable_dictionary_crosscheck.csv (if reference supplied)","literature_matrix_inventory.csv (if reference supplied)","run_manifest.json","topic_discovery_report.md"],"python_version":sys.version.split()[0],
    "package_versions":RUNTIME_INFO["packages"],"notebook_version":NOTEBOOK_VERSION,
    "runtime_information":RUNTIME_INFO,"configuration":REPRO_CONFIG,
    "candidate_outcomes_evaluated":candidate_outcomes[["candidate_outcome","status","eligible_n","predictability_feasibility_eligible"]].to_dict(orient="records"),
    "code_git_commit":CODE_GIT_COMMIT,"raw_dataset_copied_to_output_directory":False,
}
(OUTPUT_DIR / "run_manifest.json").write_text(json.dumps(manifest,indent=2,ensure_ascii=False,default=str),encoding="utf-8")
print(json.dumps(manifest,indent=2,ensure_ascii=False,default=str))


## 12 — Final audit report

Generate a concise report from this run's aggregate results. It separates observed file evidence from inference and lists human decisions. No raw records or identifiers appear in the report.

In [ ]:
def markdown_table(frame, max_rows=40):
    if frame.empty: return "_No rows available._"
    view = frame.head(max_rows).copy().fillna("not available").astype(str)
    headers = list(view.columns)
    clean = lambda value: str(value).replace("|","\\|").replace("\n"," ")
    lines = ["| " + " | ".join(clean(x) for x in headers) + " |",
             "| " + " | ".join("---" for _ in headers) + " |"]
    lines += ["| " + " | ".join(clean(x) for x in row) + " |" for row in view.itertuples(index=False,name=None)]
    return "\n".join(lines)

def pct_text(value):
    return "not estimable" if pd.isna(value) else f"{value:.2f}%"

quality_report = quality[["candidate_outcome","status","eligible_n","valid_outcome_n","missing_outcome_pct_of_eligible","positive_n","negative_n","positive_prevalence","class_imbalance_ratio_larger_over_smaller","quality_flags"]].copy()
quality_report["missing_outcome_pct_of_eligible"] = quality_report["missing_outcome_pct_of_eligible"].map(pct_text)
quality_report["positive_prevalence"] = quality_report["positive_prevalence"].map(lambda x: "not estimable" if pd.isna(x) else f"{x:.4f}")
leak_summary_report = leakage_audit.groupby(["candidate_outcome","classification"],dropna=False).size().unstack(fill_value=0).reset_index()
scoring_report = topic_evidence_matrix

human_checklist = [
    "Confirm observed v000 country/phase values, v007 year, and expected Child Recode checks in dataset_metadata.json.",
    "Review variable inventory and survey-role checks; resolve every absent or label-mismatched field.",
    "For anthropometry, verify the provisional hw13 cohort, indicator-specific score availability and plausibility bounds, the < -2 SD construction, and valid scores by hw13 status (especially WAZ with hw13=Other) against BDHS 2022 documentation.",
    "Review the anthropometry denominator bridge and validity comparison; the file and FR386 C.8 counts differ, and the cause remains unresolved.",
    "Define prediction/measurement time and the exact information available at that time; distinguish cross-sectional classification from prospective prediction.",
    "Confirm each candidate outcome definition and its candidate-specific leakage audit by both researchers before running the feasibility models.",
    "Explicitly confirm the pilot is unweighted; these preliminary metrics are not population-representative estimates.",
    "Decide observed-sample classification vs a population-risk estimand and whether survey weights enter the final analysis.",
    "Decide age/sex availability and use. Do not automatically carry pilot exclusions into the final model.",
    "Decide whether fever/diarrhea are acceptable using exact wording, reference period, and eligible universe.",
    "Resolve ARI construction from the BDHS 2022 questionnaire/report and exact value labels.",
    "Resolve anaemia field, age eligibility, adjustment, and cutoff/reference standard from BDHS 2022 documentation.",
    "Resolve national age-appropriate immunisation schedule, age, doses, source rules, and value codes.",
    "Review mortality limitations; use explicit survival/time-at-risk methods and an appropriate recode file if mortality remains in scope.",
    "Review every direct, derived, proxy-risk, and uncertain row in leakage_audit.csv and approve the preliminary prediction-feasibility feature set.",
    "Confirm PSU, strata, weight, mother, and child definitions; verify caseid+bidx uniquely identifies each birth record. The file shows v001/v021 and v022/v023 equivalence, but survey-design meanings still require BDHS documentation confirmation.",
    "Verify literature relevance against the actual papers; keep G2 provisional until full-paper verification.",
    "Record the chosen outcome, label recipe, eligibility, excluded predictors, and rationale in a dated two-person decision log before explainer experiments.",
]
recommendation_lines = "No outcome recommendation is generated; all candidates remain for human review."
checklist_lines = "\n".join(f"{i}. {item}" for i,item in enumerate(human_checklist,1))
report = f"""# BDHS 2022 Child Recode — Topic Discovery and Data Audit

**Run ID:** {RUN_ID}  \n**Experiment configuration ID:** {EXPERIMENT_CONFIG_ID}  
**Run time (UTC):** {AUDIT_TIMESTAMP_UTC}  
**Notebook version:** {NOTEBOOK_VERSION}  
**Decision status:** No thesis outcome selected or frozen.

## 1. Dataset verification

- Input: {DTA_PATH}
- File size: {FILE_SIZE_BYTES:,} bytes
- SHA-256: {DATASET_SHA256}
- Shape: {len(data):,} rows × {data.shape[1]:,} columns
- Stata file label: {DATASET_METADATA["stata_file_label"] or "not supplied"}
- v000 values: {", ".join(v000_values) if v000_values else "not available"}
- v007-derived year: {survey_year if survey_year is not None else "not verified"}
- Expected BDHS 2022 Child Recode checks passed: **{expected_bdhs_2022}**

The check uses v007 label/year, v000 Bangladesh-like country/phase code, and at least two child-recode label markers. Filename is not proof; this is not external dataset certification.

## 2. Candidate outcomes considered

{markdown_table(candidate_outcomes[["candidate_outcome","status","construction_rule","eligibility_restriction"]])}

## 3. Anthropometry measurement-status audit

{markdown_table(anthropometry_measurement_summary)}

Anthropometry eligibility is currently a provisional file-defined cohort of living children aged 0–59 months with a recognized hw13 status other than “no measurement found in household”; it is not yet the established DHS eligible denominator. DHS z-scores are stored at 100x: HAZ -600 to +600, WAZ -600 to +500, WHZ -500 to +500; the undernutrition target is score < -200. These are plausibility filters, not proof of valid measurement. Out-of-range special values are excluded. Use indicator-specific score availability rather than requiring hw13=Measured for every index: valid WAZ can occur when hw13=Other because WAZ needs weight and age, not height. Verify this interpretation against BDHS documentation and reconcile the provisional denominator with FR386. The measured subsample may be selected.

### Published anthropometry denominator comparison

FR386 Table C.8 reports 4,420 children eligible for anthropometry: 4,199 whose mothers were interviewed and 221 whose mothers were not interviewed but were in the household. Its valid-data percentages for the interviewed subgroup are 97.2% (HAZ), 97.2% (WHZ), and 98.5% (WAZ). The KR file represents children of interviewed women, so the 4,199 subgroup is the closer published comparison; the 221 not-interviewed-but-in-household subgroup is shown only to explain the full 4,420 denominator and is not merged into the KR cohort. The KR file's provisional hw13 status cohort is computed separately, and valid-score counts are split by hw13 status to expose indicator-specific exceptions. The differences are **unresolved**; the notebook does not assume maternal interview status fully explains them.

{markdown_table(denominator_bridge)}

{markdown_table(anthropometry_validity_comparison)}

The Table C.8 percentages are an external comparison only. Similar percentages do not establish identical analytic universes. See [FR386 Table C.8](https://www.dhsprogram.com/pubs/pdf/FR386/FR386.pdf).

## 4. Outcome-quality results

{markdown_table(quality_report)}

No universal missingness or balance cutoffs are asserted. Thin classes are flagged relative to the requested fold count and then checked by cluster support.

## 5. Leakage findings

Leakage classifications use labels and target construction, not correlations. See leakage_audit.csv for variable-level evidence and actions.

{markdown_table(leak_summary_report)}

## 6. Survey and cluster structure

{json.dumps(survey_structure,indent=2,ensure_ascii=False,default=str)}

No cluster, household, mother, or child identifier values are displayed. Cluster-held-out validation runs only after the team confirms PSU and mother-key interpretation; v001/v021 partition identity is reported.

## 7. Predictive Feasibility Check

{markdown_table(signal_check)}

Fold-level results (average precision, not trapezoidal PR-AUC): {markdown_table(signal_fold_metrics)}. Fold evaluation status and skipped-fold reasons are in signal_check_fold_audit.csv.

These are preliminary unweighted cluster-held-out screening metrics, not final performance or population estimates. Fold preprocessing is fit on training data only. AP baselines use each test-fold prevalence; see signal_check_fold_metrics.csv. Fold means are descriptive. If joint v001/v021 equivalence-partition grouping is used, results are a provisional feasibility diagnostic requiring human confirmation of survey-design meaning before final interpretation; no canonical PSU or outcome is selected.

## 8. Topic evidence matrix

{markdown_table(scoring_report)}

The matrix reports raw quantities and qualitative evidence only; it is not a score or ranked list. No outcome is selected.

## 9. Human review status

{recommendation_lines}

The research team must make and document the final choice after reviewing validity, feasibility, leakage, grouped support, signal, interpretability, and verified literature relevance.

## 10. Candidates pending definition review

- Mortality-related: descriptive alive/dead counts may be available, but unequal follow-up and censoring make this unsuitable as an ordinary binary prediction target without a separate survival-analysis design.
- ARI symptoms: pending verification of the BDHS 2022 wording and exact labeled coding.
- Full/age-appropriate immunisation: pending verification of Bangladesh's schedule, eligible ages, doses, and source rules.
- Child anaemia: computed only if one unique labeled status variable exists; otherwise pending measurement, adjustment, and cutoff verification.
- Any candidate with absent/mismatched labels remains pending rather than silently replaced.

## 11. Decisions requiring human confirmation

{checklist_lines}

## 12. Reproducibility

- Run ID: {RUN_ID}
- Experiment configuration ID: {EXPERIMENT_CONFIG_ID}
- Configuration identity SHA-256: {CONFIGURATION_IDENTITY_SHA256}
- Dataset SHA-256: {DATASET_SHA256}
- Random seed: {RANDOM_SEED}
- Python: {sys.version.split()[0]}
- Package versions: {json.dumps(RUNTIME_INFO["packages"],sort_keys=True)}
- Git commit: {manifest["code_git_commit"] or "not available"}
- Manifest: run_manifest.json
- Raw dataset copied to outputs: **No**
- Explainability methods used: **None**

## Sources and definition checks

- [Bangladesh DHS 2022 Final Report (FR386)](https://www.dhsprogram.com/pubs/pdf/FR386/FR386.pdf)
- [DHS Guide to Statistics](https://preview.dhsprogram.com/pubs/pdf/DHSG1/Guide_to_DHS_Statistics_DHS-7.pdf)
- [DHS Program indicator code for child vaccination](https://github.com/DHSProgram/DHS-Indicators-R/blob/main/Chap10_CH/CH_VAC.R)

Use the actual BDHS report, questionnaire, dictionary, and file labels for survey-specific definitions; general DHS code is a reference, not proof of this file's coding.
"""
(OUTPUT_DIR / "topic_discovery_report.md").write_text(report,encoding="utf-8")
print(f"Saved aggregate-only outputs in: {OUTPUT_DIR.resolve()}")
print(f"Report: {(OUTPUT_DIR / 'topic_discovery_report.md').resolve()}")
print(f"Checklist items requiring human confirmation: {len(human_checklist)}")
display(topic_evidence_matrix[["candidate_outcome","team_decision","predictability_feasibility"]])


## 13 — Cross-check the supplied reference files

This section checks whether the separate variable dictionary agrees with the labels embedded in the DTA and imports the literature matrix as a **working evidence inventory**. It does not independently verify the papers, their claims, or the novelty of G1/G2. Missing reference files are reported as unavailable rather than silently substituted.

In [ ]:
# Optional reference-file checks. Put the supplied CSVs in the configured reference folder or edit the paths in cell 3.
source_audit = {}

# A. Variable dictionary cross-check: compare variable coverage, variable labels, and value-label counts.
if VARIABLE_DICTIONARY_PATH.is_file():
    ref_dict = pd.read_csv(VARIABLE_DICTIONARY_PATH, dtype={"variable":"string"})
    required_cols = {"variable", "label", "n_value_labels"}
    missing_cols = required_cols - set(ref_dict.columns)
    if missing_cols:
        source_audit["variable_dictionary"] = {"status":"invalid schema", "missing_columns":sorted(missing_cols)}
    else:
        ref_dict = ref_dict.dropna(subset=["variable"]).copy()
        ref_dict["variable"] = ref_dict["variable"].astype(str)
        duplicate_vars = sorted(ref_dict.loc[ref_dict["variable"].duplicated(keep=False), "variable"].unique().tolist())
        dta_labels = getattr(stata_meta, "column_names_to_labels", {}) or getattr(stata_meta, "variable_to_label", {}) or {}
        dta_variable_value_labels = getattr(stata_meta, "variable_value_labels", {}) or {}
        def _norm_code(value):
            try:
                number = float(str(value).strip())
                return str(int(number)) if number.is_integer() else str(number)
            except (TypeError, ValueError):
                return str(value).strip()
        def _norm_label(value):
            return re.sub(r"\s+", " ", str(value).strip()).casefold()
        def _parse_csv_value_labels(raw):
            parsed = {}
            if pd.isna(raw) or not str(raw).strip():
                return parsed
            for part in str(raw).split(";"):
                code, separator, label_text = part.partition("=")
                if not separator:
                    continue
                parsed[_norm_code(code)] = _norm_label(label_text)
            return parsed
        rows = []
        for _, rec in ref_dict.iterrows():
            var = rec["variable"]
            in_dta = var in data.columns
            dta_label = str(dta_labels.get(var, "") or "") if in_dta else "NOT PRESENT"
            csv_label = "" if pd.isna(rec.get("label", "")) else str(rec.get("label", "")).strip()
            label_match = (re.sub(r"\s+", " ", dta_label.strip()).casefold() ==
                           re.sub(r"\s+", " ", csv_label.strip()).casefold()) if in_dta else False
            dta_vl = dta_variable_value_labels.get(var, {}) if isinstance(dta_variable_value_labels, dict) else {}
            dta_vl_count = len(dta_vl) if isinstance(dta_vl, dict) else 0
            csv_vl_count = int(pd.to_numeric(pd.Series([rec.get("n_value_labels")]), errors="coerce").fillna(0).iloc[0])
            csv_vl_map = _parse_csv_value_labels(rec.get("value_labels")) if "value_labels" in ref_dict.columns else {}
            dta_vl_map = {_norm_code(k):_norm_label(v) for k,v in dta_vl.items()} if isinstance(dta_vl, dict) else {}
            mapping_comparable = "value_labels" in ref_dict.columns and in_dta
            mapping_match = (csv_vl_map == dta_vl_map) if mapping_comparable else False
            rows.append({"variable":var,"present_in_dta":bool(in_dta),"csv_label":csv_label,
                         "dta_label":dta_label,"label_match":bool(label_match),
                         "csv_value_label_count":csv_vl_count,"dta_value_label_count":int(dta_vl_count),
                         "value_label_count_match":bool(in_dta and csv_vl_count==dta_vl_count),
                         "value_label_mapping_comparable":bool(mapping_comparable),
                         "value_label_mapping_match":bool(mapping_match) if mapping_comparable else None})
        dictionary_crosscheck = pd.DataFrame(rows)
        dictionary_crosscheck.to_csv(OUTPUT_DIR/"variable_dictionary_crosscheck.csv",index=False)
        dta_not_in_csv = sorted(set(map(str,data.columns)) - set(ref_dict["variable"].astype(str)))
        source_audit["variable_dictionary"] = {
            "status":"compared","reference_rows":int(len(ref_dict)),"duplicate_variable_names":duplicate_vars,
            "reference_variables_missing_from_dta":int((~dictionary_crosscheck["present_in_dta"]).sum()),
            "dta_variables_missing_from_reference_csv":len(dta_not_in_csv),
            "variable_label_mismatches":int((~dictionary_crosscheck["label_match"]).sum()),
            "value_label_count_mismatches":int((~dictionary_crosscheck["value_label_count_match"]).sum()),
            "value_label_mapping_mismatches":int((dictionary_crosscheck["value_label_mapping_match"] == False).sum()),
            "dta_variables_missing_from_reference_csv_names":dta_not_in_csv[:100],
            "note":"Variable labels and value-label code-to-text mappings are normalized and compared. Review all mismatches; differences can reflect extraction/formatting as well as substantive metadata drift."
        }
else:
    source_audit["variable_dictionary"] = {"status":"not available","expected_path":str(VARIABLE_DICTIONARY_PATH)}

# B. Literature matrix inventory: retain supplied verification/read-level status; do not upgrade it.
if LITERATURE_MATRIX_PATH.is_file():
    lit = pd.read_csv(LITERATURE_MATRIX_PATH)
    required_lit = {"ID","Title","Year","Read level","Explanation method","Did they compare explanation?","Dataset","Verification status (2026-10-16)"}
    missing_lit = required_lit - set(lit.columns)
    if missing_lit:
        source_audit["literature_matrix"] = {"status":"invalid schema","missing_columns":sorted(missing_lit)}
    else:
        keep_cols = [c for c in ["ID","Title","Authors","Year","Venue","DOI/Link","Read level",
                    "Explanation method","Did they compare explanation?","Dataset","Limitations",
                    "What is missing","What our thesis can add/improve","Verification status (2026-10-16)"] if c in lit.columns]
        lit[keep_cols].to_csv(OUTPUT_DIR/"literature_matrix_inventory.csv",index=False)
        status_counts = lit["Verification status (2026-10-16)"].fillna("not recorded").value_counts().to_dict()
        read_counts = lit["Read level"].fillna("not recorded").value_counts().to_dict()
        source_audit["literature_matrix"] = {
            "status":"imported as working inventory; not independently verified",
            "paper_records":int(len(lit)),"verification_status_counts":{str(k):int(v) for k,v in status_counts.items()},
            "read_level_counts":{str(k):int(v) for k,v in read_counts.items()},
            "note":"Use this inventory to prioritize Track A (G1 overlap) and Track B (DHS/BDHS application) full-paper checks. Do not infer novelty or claim support from matrix text alone."
        }
else:
    source_audit["literature_matrix"] = {"status":"not available","expected_path":str(LITERATURE_MATRIX_PATH)}

(OUTPUT_DIR/"reference_source_audit.json").write_text(json.dumps(source_audit,indent=2,ensure_ascii=False,default=str),encoding="utf-8")
print(json.dumps(source_audit,indent=2,ensure_ascii=False,default=str))
print("Reference-file checks saved. Full value-label text and literature claims still require manual/full-source verification.")

# Append the source-audit result to the main report, which was generated in the previous section.
report_path = OUTPUT_DIR / "topic_discovery_report.md"
if report_path.exists():
    report_appendix = "\n\n## 13. Supplied reference-file audit\n\n```json\n" + json.dumps(source_audit, indent=2, ensure_ascii=False, default=str) + "\n```\n\nThe variable dictionary comparison checks labels and value-label counts; it does not compare every code-to-text mapping. The literature matrix is a working inventory only. Its status fields are carried forward as supplied and do not independently verify paper claims, full-text coverage, or novelty.\n"
    with open(report_path, "a", encoding="utf-8") as handle:
        handle.write(report_appendix)
    print(f"Appended reference-file audit to: {report_path}")

# Refresh the manifest after the optional reference-source files have been processed.
manifest_path = OUTPUT_DIR / "run_manifest.json"
if manifest_path.exists():
    final_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    final_manifest["reference_input_files"] = REFERENCE_INPUT_FILES
    final_manifest["reference_source_audit"] = source_audit
    for generated in ["reference_source_audit.json", "variable_dictionary_crosscheck.csv", "literature_matrix_inventory.csv", "signal_check_fold_audit.csv"]:
        generated_path = OUTPUT_DIR / generated
        if generated_path.exists() and generated not in final_manifest.get("output_files", []):
            final_manifest.setdefault("output_files", []).append(generated)
    manifest_path.write_text(json.dumps(final_manifest,indent=2,ensure_ascii=False,default=str),encoding="utf-8")
    print(f"Updated run manifest with reference inputs and final output inventory: {manifest_path}")


## Final team checklist before freezing the prediction task

Bring back aggregate outputs only and resolve these before explanation experiments:

1. Actual v000/v007 labels and values, dataset hash, and Child Recode verification.
2. Review reference_source_audit.json. Resolve dictionary mismatches; do not treat the literature matrix as independently verified.
3. Human-confirmed PSU, strata, weight, mother, and child definitions; review v001/v021 PSU and v022/v023 strata exact values and partition identity.
4. hw13 status summary, indicator-specific score availability/bounds, valid/eligible N, valid scores by hw13 status (especially WAZ with hw13='Other'), and denominator reconciliation; document measured-subsample selection.
5. For each candidate: actual labels/value labels, eligibility, invalid/missing handling, class counts, prevalence, imbalance, and leakage exclusions.
6. Resolve fever/diarrhoea, ARI, anaemia, immunisation, and mortality definitions; unresolved candidates stay pending.
7. Review pilot predictor policy and ambiguous dictionary matches. Define prediction time and age/sex use; pilot exclusions do not automatically carry into final models.
8. Confirm PSU/mother-key structure before interpreting grouped results.
9. Review fold-level ROC-AUC and average precision (AP), each fold's AP prevalence baseline, and the unweighted preliminary-screening limitation.
10. Confirm prediction/measurement time, observed-sample classification versus population-risk estimand, and the explicit unweighted pilot decision.
11. Verify literature relevance; G2 remains provisional.
12. Record a dated two-person task decision. This notebook does not choose an outcome.

**The predictive-feasibility screen is intentionally skipped until the explicit confirmations in cell 3 are complete. This notebook supplies evidence. It does not choose the outcome.**


## Two-person sign-off record (complete outside the automated audit)

Before moving to the explanation experiments, both researchers must record and agree on:

- Exact outcome definition and label recipe, including eligible ages, alive/deceased handling, invalid codes and missingness.
- Prediction-time boundary and which predictors are genuinely available at that time.
- Population/estimand: observed-sample prediction or population-risk estimation.
- PSU and strata definitions from BDHS documentation; mother/child grouping key; whether train/test partitions must be cluster-disjoint.
- Survey-weight policy for training and evaluation, with the reason for the chosen estimand.
- Final predictor list and explicit exclusion of direct/derived leakage and any proxy variables the team decides are inappropriate.
- Model list and explanation compatibility. The three-way SHAP/LIME/IG comparison is restricted to differentiable models; Random Forest is SHAP-vs-LIME only unless a justified differentiable surrogate is separately evaluated.
- A dated decision-log entry signed by both team members.

**Do not start the explainer experiment by simply setting an outcome variable in code.** First resolve the items above and update the frozen research framework's decision log.